In [ ]:
"""
BoR-as-Reward on SciFact (BEIR). Single Colab cell.
pip install rank_bm25 beir
"""
import subprocess, sys, random, math, numpy as np
from collections import defaultdict
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rank_bm25", "beir"])
from rank_bm25 import BM25Okapi
from beir import util
from beir.datasets.data_loader import GenericDataLoader

random.seed(42); np.random.seed(42)

# ── Load SciFact ──
url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip"
data_path = util.download_and_unzip(url, "datasets")
corpus, queries, qrels = GenericDataLoader(data_path).load(split="test")

doc_ids = list(corpus.keys())
doc_texts = [corpus[d]["title"] + " " + corpus[d]["text"] for d in doc_ids]
doc_id_to_idx = {d: i for i, d in enumerate(doc_ids)}
N = len(doc_ids)

query_list = []
for qid in queries:
    if qid not in qrels: continue
    rel = set(doc_id_to_idx[d] for d, s in qrels[qid].items() if s > 0 and d in doc_id_to_idx)
    if rel:
        query_list.append({'qid': qid, 'text': queries[qid], 'rel': rel, 'R_q': len(rel)})

# ── BM25 rankings ──
bm25 = BM25Okapi([d.lower().split() for d in doc_texts])
for q in query_list:
    scores = bm25.get_scores(q['text'].lower().split())
    q['ranked'] = sorted(enumerate(scores), key=lambda x: -x[1])[:200]

random.shuffle(query_list)
train_q, test_q = query_list[:int(len(query_list)*0.7)], query_list[int(len(query_list)*0.7):]
print(f"SciFact: {N} docs, {len(train_q)} train, {len(test_q)} test queries\n")

# ── Environment ──
class Env:
    def __init__(self, queries, N, dk=5, max_k=100):
        self.queries, self.N, self.dk, self.max_k = queries, N, dk, max_k
    def new_query(self):
        self.q = random.choice(self.queries); self.k = 0; self.found = False; self.gap = 0
        return self._s()
    def step(self, a):
        if a == 0: return self._s(), self._r(), True
        self.k = min(self.k + self.dk, self.max_k)
        ret = self.q['ranked'][:self.k]
        self.found = any(i in self.q['rel'] for i, _ in ret)
        if ret:
            sc = [s for _, s in ret]
            t5, rest = sc[:min(5,len(sc))], sc[min(5,len(sc)):min(20,len(sc))]
            self.gap = np.mean(t5) - np.mean(rest) if rest else 0
        if self.k >= self.max_k: return self._s(), self._r(), True
        return self._s(), None, False
    def _s(self):
        d = min(self.k // self.dk, 19)
        g = 2 if self.gap > 3 else (1 if self.gap > 1 else 0)
        lam = self.k * self.q['R_q'] / self.N if self.k > 0 else 0
        l = 2 if lam > 0.5 else (1 if lam > 0.1 else 0)
        return (d, g, l, int(self.found))
    def _r(self):
        k = max(self.k, 1)
        p = max(1e-12, 1 - ((1 - self.q['R_q']/self.N) ** k))
        return {'bor': max(0.01, -math.log2(p)) if self.found else 0.0,
                'f1': 1.0 if self.found else 0.0, 'p_rand': p, 'k': k, 'found': self.found}

# ── Q-learning ──
class QL:
    def __init__(self):
        self.Q = defaultdict(lambda: [0.0, 0.0]); self.a=0.1; self.g=0.95; self.e=0.4
    def act(self, s, greedy=False):
        if not greedy and random.random()<self.e: return random.randint(0,1)
        return 0 if self.Q[s][0]>=self.Q[s][1] else 1
    def update(self, s, a, r, s2, done):
        nxt = max(self.Q[s2]) if not done else 0
        self.Q[s][a] += self.a * (r + self.g * nxt - self.Q[s][a])

def train(env, rk, n=60000, sc=0.01):
    ag = QL()
    for ep in range(n):
        if ep > n*0.7: ag.e=0.03
        elif ep > n*0.4: ag.e=0.15
        s = env.new_query()
        while True:
            a = ag.act(s)
            s2, rw, done = env.step(a)
            if done: ag.update(s, a, rw[rk], s2, True); break
            else: ag.update(s, a, -sc, s2, False); s = s2
    return ag

def evaluate(env, agent, qs, runs=5):
    res = []
    for _ in range(runs):
        for q in qs:
            env.queries = [q]; s = env.new_query()
            while True:
                a = agent.act(s, greedy=True)
                s2, rw, done = env.step(a)
                if done: res.append(rw); break
                s = s2
    return res

def eval_fixed(env, qs, target_k):
    res = []
    for q in qs:
        env.queries = [q]; env.new_query()
        for _ in range(target_k // env.dk): env.step(1)
        _, rw, _ = env.step(0); res.append(rw)
    return res

# ── Train & Evaluate ──
tr_env = Env(train_q, N); te_env = Env(test_q, N)

print("Training...", end=" ", flush=True)
bor_ag = train(tr_env, 'bor'); f1_ag = train(tr_env, 'f1')
print("done\n")

br = evaluate(te_env, bor_ag, test_q)
fr = evaluate(te_env, f1_ag, test_q)
f10 = eval_fixed(te_env, test_q, 10)
f20 = eval_fixed(te_env, test_q, 20)
f50 = eval_fixed(te_env, test_q, 50)

# ── Results ──
def row(res, label):
    k=np.mean([r['k'] for r in res]); f=np.mean([r['found'] for r in res])*100
    b=np.mean([r['bor'] for r in res if r['found']] or [0])
    print(f"  {label:<20} K={k:>5.1f}  Found={f:>5.1f}%  BoR={b:>5.2f}  eff={f/k:>.2f}%/doc")

print(f"{'─'*70}")
row(br, "BoR agent"); row(fr, "F1 agent")
row(f10, "Fixed K=10"); row(f20, "Fixed K=20"); row(f50, "Fixed K=50")
print(f"{'─'*70}")
print(f"  BoR K std: {np.std([r['k'] for r in br]):.2f}   F1 K std: {np.std([r['k'] for r in fr]):.2f}")

/usr/local/lib/python3.12/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


datasets/scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

  0%|          | 0/5183 [00:00<?, ?it/s]

SciFact: 5183 docs, 210 train, 90 test queries

Training... done

──────────────────────────────────────────────────────────────────────
  BoR agent            K=  7.2  Found= 78.9%  BoR= 9.76  eff=11.01%/doc
  F1 agent             K=  5.0  Found= 66.7%  BoR= 9.94  eff=13.33%/doc
  Fixed K=10           K= 10.0  Found= 75.6%  BoR= 8.94  eff=7.56%/doc
  Fixed K=20           K= 20.0  Found= 81.1%  BoR= 7.94  eff=4.06%/doc
  Fixed K=50           K= 50.0  Found= 85.6%  BoR= 6.63  eff=1.71%/doc
──────────────────────────────────────────────────────────────────────
  BoR K std: 3.34   F1 K std: 0.00


In [ ]:
"""
BoR-as-Reward: DQN on NFCorpus. Single Colab cell.
pip install rank_bm25 beir torch
"""
import subprocess, sys, random, math, numpy as np
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rank_bm25", "beir", "torch"])

import torch
import torch.nn as nn
import torch.optim as optim
from collections import defaultdict, deque
from rank_bm25 import BM25Okapi
from beir import util
from beir.datasets.data_loader import GenericDataLoader

random.seed(42); np.random.seed(42); torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Load NFCorpus ──
url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip"
data_path = util.download_and_unzip(url, "datasets")
corpus, queries, qrels = GenericDataLoader(data_path).load(split="test")

doc_ids = list(corpus.keys())
doc_texts = [corpus[d]["title"] + " " + corpus[d]["text"] for d in doc_ids]
doc_id_to_idx = {d: i for i, d in enumerate(doc_ids)}
N = len(doc_ids)

query_list = []
for qid in queries:
    if qid not in qrels: continue
    rel = set(doc_id_to_idx[d] for d, s in qrels[qid].items() if s > 0 and d in doc_id_to_idx)
    if rel:
        query_list.append({'qid': qid, 'text': queries[qid], 'rel': rel, 'R_q': len(rel)})

rqs = [q['R_q'] for q in query_list]
print(f"NFCorpus: {N} docs, {len(query_list)} queries")
print(f"R_q: min={min(rqs)} max={max(rqs)} mean={np.mean(rqs):.1f} median={np.median(rqs):.0f}")

# ── BM25 ──
bm25 = BM25Okapi([d.lower().split() for d in doc_texts])
for q in query_list:
    scores = bm25.get_scores(q['text'].lower().split())
    q['ranked'] = sorted(enumerate(scores), key=lambda x: -x[1])[:200]

random.shuffle(query_list)
train_q, test_q = query_list[:int(len(query_list)*0.7)], query_list[int(len(query_list)*0.7):]
print(f"Train: {len(train_q)}, Test: {len(test_q)}\n")

# ── Environment with CONTINUOUS state ──
class Env:
    def __init__(self, queries, N, dk=5, max_k=100):
        self.queries, self.N, self.dk, self.max_k = queries, N, dk, max_k

    def new_query(self):
        self.q = random.choice(self.queries)
        self.k = 0; self.found = False
        self.top_score = 0; self.gap = 0; self.score_std = 0
        return self._s()

    def step(self, a):
        if a == 0: return self._s(), self._r(), True
        self.k = min(self.k + self.dk, self.max_k)
        ret = self.q['ranked'][:self.k]
        self.found = any(i in self.q['rel'] for i, _ in ret)
        if ret:
            sc = [s for _, s in ret]
            self.top_score = sc[0]
            t5 = sc[:min(5, len(sc))]
            rest = sc[min(5, len(sc)):min(20, len(sc))]
            self.gap = np.mean(t5) - np.mean(rest) if rest else 0
            self.score_std = np.std(sc)
        if self.k >= self.max_k: return self._s(), self._r(), True
        return self._s(), None, False

    def _s(self):
        """Continuous state vector: 7 features."""
        k = max(self.k, 1)
        lam = k * self.q['R_q'] / self.N
        p_rand = 1 - ((1 - self.q['R_q'] / self.N) ** k)
        bor_ceil = -math.log2(max(p_rand, 1e-12))
        return np.array([
            self.k / self.max_k,         # normalized depth
            self.top_score / 20.0,       # normalized top BM25 score
            self.gap / 10.0,             # normalized score gap
            self.score_std / 10.0,       # score spread
            min(lam, 5.0) / 5.0,        # normalized lambda (capped)
            bor_ceil / 12.0,             # normalized BoR ceiling
            float(self.found),           # found indicator
        ], dtype=np.float32)

    def _r(self):
        k = max(self.k, 1)
        p = max(1e-12, 1 - ((1 - self.q['R_q'] / self.N) ** k))
        return {'bor': max(0.01, -math.log2(p)) if self.found else 0.0,
                'f1': 1.0 if self.found else 0.0,
                'p_rand': p, 'k': k, 'found': self.found, 'R_q': self.q['R_q']}

# ── DQN ──
class DQN(nn.Module):
    def __init__(self, state_dim=7, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 2)  # Q(STOP), Q(CONTINUE)
        )
    def forward(self, x): return self.net(x)

class ReplayBuffer:
    def __init__(self, cap=20000):
        self.buf = deque(maxlen=cap)
    def push(self, *args): self.buf.append(args)
    def sample(self, n):
        batch = random.sample(self.buf, min(n, len(self.buf)))
        s, a, r, s2, d = zip(*batch)
        return (torch.FloatTensor(np.array(s)).to(device),
                torch.LongTensor(a).to(device),
                torch.FloatTensor(r).to(device),
                torch.FloatTensor(np.array(s2)).to(device),
                torch.FloatTensor(d).to(device))
    def __len__(self): return len(self.buf)

def train_dqn(env, reward_key, n_eps=15000, step_cost=0.01,
              batch_size=128, gamma=0.95, lr=1e-3):
    policy = DQN().to(device)
    target = DQN().to(device)
    target.load_state_dict(policy.state_dict())
    opt = optim.Adam(policy.parameters(), lr=lr)
    buf = ReplayBuffer()
    eps = 0.5

    for ep in range(n_eps):
        if ep > n_eps * 0.7: eps = 0.03
        elif ep > n_eps * 0.4: eps = 0.1

        s = env.new_query()
        while True:
            # Epsilon-greedy
            if random.random() < eps:
                a = random.randint(0, 1)
            else:
                with torch.no_grad():
                    q_vals = policy(torch.FloatTensor(s).unsqueeze(0).to(device))
                    a = q_vals.argmax(1).item()

            s2, rw, done = env.step(a)
            r = rw[reward_key] if done else -step_cost
            buf.push(s, a, r, s2, float(done))
            s = s2

            # Train
            if len(buf) >= batch_size:
                sb, ab, rb, s2b, db = buf.sample(batch_size)
                q_cur = policy(sb).gather(1, ab.unsqueeze(1)).squeeze(1)
                with torch.no_grad():
                    q_next = target(s2b).max(1)[0]
                    q_target = rb + gamma * q_next * (1 - db)
                loss = nn.MSELoss()(q_cur, q_target)
                opt.zero_grad(); loss.backward(); opt.step()

            if done: break

        # Update target network
        if ep % 500 == 0:
            target.load_state_dict(policy.state_dict())
        if (ep+1) % 3000 == 0:
            print(f"{ep+1}/{n_eps}", end=" ", flush=True)

    return policy

def evaluate(env, policy, qs, runs=5):
    res = []
    for _ in range(runs):
        for q in qs:
            env.queries = [q]; s = env.new_query()
            while True:
                with torch.no_grad():
                    qv = policy(torch.FloatTensor(s).unsqueeze(0).to(device))
                    a = qv.argmax(1).item()
                s2, rw, done = env.step(a)
                if done: res.append(rw); break
                s = s2
    return res

def eval_fixed(env, qs, target_k):
    res = []
    for q in qs:
        env.queries = [q]; env.new_query()
        for _ in range(target_k // env.dk): env.step(1)
        _, rw, _ = env.step(0); res.append(rw)
    return res

# ── Train ──
tr_env = Env(train_q, N); te_env = Env(test_q, N)

print("Training BoR DQN...", end=" ", flush=True)
bor_pol = train_dqn(tr_env, 'bor', n_eps=15000)
print("done")

print("Training F1 DQN...", end=" ", flush=True)
f1_pol = train_dqn(tr_env, 'f1', n_eps=15000)
print("done\n")

# ── Evaluate ──
br = evaluate(te_env, bor_pol, test_q)
fr = evaluate(te_env, f1_pol, test_q)
f10 = eval_fixed(te_env, test_q, 10)
f20 = eval_fixed(te_env, test_q, 20)
f50 = eval_fixed(te_env, test_q, 50)

def row(res, label):
    k=np.mean([r['k'] for r in res]); f=np.mean([r['found'] for r in res])*100
    b=np.mean([r['bor'] for r in res if r['found']] or [0])
    print(f"  {label:<20} K={k:>5.1f}  Found={f:>5.1f}%  BoR={b:>5.2f}  eff={f/k:>.2f}%/doc")

print(f"{'─'*70}")
row(br, "BoR DQN"); row(fr, "F1 DQN")
row(f10, "Fixed K=10"); row(f20, "Fixed K=20"); row(f50, "Fixed K=50")
print(f"{'─'*70}")
print(f"  BoR K std: {np.std([r['k'] for r in br]):.2f}   F1 K std: {np.std([r['k'] for r in fr]):.2f}")

# ── Per-R_q breakdown ──
print(f"\n  BoR DQN: K by R_q bin")
print(f"  {'R_q bin':<15} {'Avg K':>7} {'Found%':>8} {'BoR':>7}")
print(f"  {'─'*40}")
for label, lo, hi in [('R_q≤5', 0, 5), ('5<R_q≤30', 6, 30), ('R_q>30', 31, 9999)]:
    items = [r for r in br if lo <= r['R_q'] <= hi]
    if not items: continue
    print(f"  {label:<15} {np.mean([r['k'] for r in items]):>7.1f} "
          f"{np.mean([r['found'] for r in items])*100:>7.1f}% "
          f"{np.mean([r['bor'] for r in items if r['found']] or [0]):>7.2f}")

print(f"\n  F1 DQN: K by R_q bin")
print(f"  {'R_q bin':<15} {'Avg K':>7} {'Found%':>8} {'BoR':>7}")
print(f"  {'─'*40}")
for label, lo, hi in [('R_q≤5', 0, 5), ('5<R_q≤30', 6, 30), ('R_q>30', 31, 9999)]:
    items = [r for r in fr if lo <= r['R_q'] <= hi]
    if not items: continue
    print(f"  {label:<15} {np.mean([r['k'] for r in items]):>7.1f} "
          f"{np.mean([r['found'] for r in items])*100:>7.1f}% "
          f"{np.mean([r['bor'] for r in items if r['found']] or [0]):>7.2f}")

  0%|          | 0/3633 [00:00<?, ?it/s]

NFCorpus: 3633 docs, 323 queries
R_q: min=1 max=475 mean=38.2 median=16
Train: 226, Test: 97

Training BoR DQN... 3000/15000 6000/15000 9000/15000 12000/15000 15000/15000 done
Training F1 DQN... 3000/15000 6000/15000 9000/15000 12000/15000 15000/15000 done

──────────────────────────────────────────────────────────────────────
  BoR DQN              K= 22.9  Found= 71.1%  BoR= 4.90  eff=3.10%/doc
  F1 DQN               K= 24.5  Found= 68.0%  BoR= 4.88  eff=2.77%/doc
  Fixed K=10           K= 10.0  Found= 60.8%  BoR= 4.48  eff=6.08%/doc
  Fixed K=20           K= 20.0  Found= 62.9%  BoR= 3.54  eff=3.14%/doc
  Fixed K=50           K= 50.0  Found= 69.1%  BoR= 2.30  eff=1.38%/doc
──────────────────────────────────────────────────────────────────────
  BoR K std: 30.53   F1 K std: 31.92

  BoR DQN: K by R_q bin
  R_q bin           Avg K   Found%     BoR
  ────────────────────────────────────────
  R_q≤5              29.0    46.7%    7.77
  5<R_q≤30           28.2    73.0%    5.21
  R_q>30   

In [ ]:
"""
BoR-as-Reward: DQN on MS MARCO Passage.
Uses ir_datasets (pure Python, no Java) + rank_bm25.
Local BM25 index of sampled passages; BoR computed against full N=8.8M.
"""
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ir_datasets", "rank_bm25", "torch"])

import random, math, numpy as np, ir_datasets, torch
import torch.nn as nn, torch.optim as optim
from collections import defaultdict, deque
from rank_bm25 import BM25Okapi

random.seed(42); np.random.seed(42); torch.manual_seed(42)

# ── Load (cached) ──
dataset = ir_datasets.load("msmarco-passage/dev/small")

qrels = defaultdict(set)
for qrel in dataset.qrels_iter():
    if qrel.relevance > 0:
        qrels[qrel.query_id].add(qrel.doc_id)

queries = {}
for q in dataset.queries_iter():
    if q.query_id in qrels:
        queries[q.query_id] = q.text
    if len(queries) >= 1000: break

all_rel_ids = set()
for qid in queries:
    all_rel_ids.update(qrels[qid])

# ── Load MORE docs this time (50K target) ──
print("Loading documents (50K target)...", flush=True)
docs_store = {}
doc_count = 0
for doc in dataset.docs_iter():
    if doc.doc_id in all_rel_ids:
        docs_store[doc.doc_id] = doc.text
    elif len(docs_store) < 50000 and random.random() < 0.05:
        docs_store[doc.doc_id] = doc.text
    doc_count += 1
    if doc_count % 2000000 == 0:
        print(f"  scanned {doc_count/1e6:.0f}M, stored {len(docs_store)}", flush=True)

N_LOCAL = len(docs_store)
N_FULL = 8841823
print(f"Local index: {N_LOCAL} docs\n")

doc_ids_list = list(docs_store.keys())
doc_id_to_idx = {did: i for i, did in enumerate(doc_ids_list)}
tokenized = [docs_store[did].lower().split() for did in doc_ids_list]

print("Building BM25 index...", end=" ", flush=True)
bm25 = BM25Okapi(tokenized)
print("done")

# ── Rankings ──
print("Computing BM25 rankings...", flush=True)
query_list = []
qids = list(queries.keys())[:500]

for i, qid in enumerate(qids):
    scores = bm25.get_scores(queries[qid].lower().split())
    ranked_idx = sorted(enumerate(scores), key=lambda x: -x[1])[:200]
    rel_local = set(doc_id_to_idx[d] for d in qrels[qid] if d in doc_id_to_idx)
    R_q = len(qrels[qid])
    ranked = []; first_hit = None
    for rank, (idx, score) in enumerate(ranked_idx):
        is_rel = idx in rel_local
        ranked.append((idx, score, is_rel))
        if is_rel and first_hit is None: first_hit = rank + 1
    query_list.append({'qid': qid, 'R_q': max(R_q,1), 'ranked': ranked, 'first_hit': first_hit})
    if (i+1) % 100 == 0: print(f"  {i+1}/{len(qids)}", flush=True)

print(f"\n{len(query_list)} queries, local index {N_LOCAL}, BoR against N={N_FULL:,}")
for k in [5, 10, 20, 50]:
    found = sum(1 for q in query_list if q['first_hit'] and q['first_hit'] <= k)
    p_rand = np.mean([1-((1-q['R_q']/N_FULL)**k) for q in query_list])
    print(f"  K={k:>3}: found={found/len(query_list)*100:>5.1f}%  BoR_ceil={-math.log2(max(p_rand,1e-20)):.1f} bits")

random.shuffle(query_list)
split = int(len(query_list) * 0.7)
train_q, test_q = query_list[:split], query_list[split:]
print(f"Train: {len(train_q)}, Test: {len(test_q)}\n")

# ── Environment ──
class Env:
    def __init__(self, queries, N, dk=5, max_k=100):
        self.queries, self.N, self.dk, self.max_k = queries, N, dk, max_k
    def new_query(self):
        self.q = random.choice(self.queries)
        self.k=0; self.found=False; self.top_score=0; self.gap=0; self.score_std=0
        return self._s()
    def step(self, a):
        if a == 0: return self._s(), self._r(), True
        self.k = min(self.k + self.dk, self.max_k)
        ranked = self.q['ranked'][:self.k]
        self.found = any(r for _,_,r in ranked)
        if ranked:
            sc = [s for _,s,_ in ranked]
            self.top_score = sc[0]
            t5, rest = sc[:min(5,len(sc))], sc[min(5,len(sc)):min(20,len(sc))]
            self.gap = np.mean(t5) - np.mean(rest) if rest else 0
            self.score_std = np.std(sc)
        if self.k >= self.max_k: return self._s(), self._r(), True
        return self._s(), None, False
    def _s(self):
        k = max(self.k, 1)
        lam = k * self.q['R_q'] / self.N
        p_rand = max(1e-20, 1-((1-self.q['R_q']/self.N)**k))
        bor_ceil = -math.log2(p_rand)
        return np.array([
            self.k / self.max_k,
            self.top_score / 20.0,
            self.gap / 5.0,
            self.score_std / 5.0,
            min(lam * 1e5, 5.0) / 5.0,
            min(bor_ceil, 25.0) / 25.0,
            float(self.found),
        ], dtype=np.float32)
    def _r(self):
        k = max(self.k, 1)
        p = max(1e-20, 1-((1-self.q['R_q']/self.N)**k))
        return {'bor': max(0.01, -math.log2(p)) if self.found else 0.0,
                'f1': 1.0 if self.found else 0.0,
                'p_rand': p, 'k': k, 'found': self.found, 'R_q': self.q['R_q']}

# ── DQN ──
class DQN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(7,64), nn.ReLU(), nn.Linear(64,64), nn.ReLU(), nn.Linear(64,2))
    def forward(self, x): return self.net(x)

class Buf:
    def __init__(self, cap=30000):
        self.b = deque(maxlen=cap)
    def push(self, *a): self.b.append(a)
    def sample(self, n):
        batch = random.sample(self.b, min(n, len(self.b)))
        s,a,r,s2,d = zip(*batch)
        return (torch.FloatTensor(np.array(s)), torch.LongTensor(a),
                torch.FloatTensor(r), torch.FloatTensor(np.array(s2)), torch.FloatTensor(d))
    def __len__(self): return len(self.b)

def train_dqn(env, rk, n_eps=20000, sc=0.01, bs=128, gamma=0.95):
    pol=DQN(); tgt=DQN(); tgt.load_state_dict(pol.state_dict())
    opt=optim.Adam(pol.parameters(), lr=1e-3); buf=Buf(); eps=0.5
    for ep in range(n_eps):
        if ep>n_eps*0.7: eps=0.03
        elif ep>n_eps*0.4: eps=0.1
        s=env.new_query()
        while True:
            if random.random()<eps: a=random.randint(0,1)
            else:
                with torch.no_grad(): a=pol(torch.FloatTensor(s).unsqueeze(0)).argmax(1).item()
            s2,rw,done=env.step(a)
            r=rw[rk] if done else -sc
            buf.push(s,a,r,s2,float(done)); s=s2
            if len(buf)>=bs:
                sb,ab,rb,s2b,db=buf.sample(bs)
                qc=pol(sb).gather(1,ab.unsqueeze(1)).squeeze(1)
                with torch.no_grad(): qt=rb+gamma*tgt(s2b).max(1)[0]*(1-db)
                loss=nn.MSELoss()(qc,qt); opt.zero_grad(); loss.backward(); opt.step()
            if done: break
        if ep%500==0: tgt.load_state_dict(pol.state_dict())
        if (ep+1)%5000==0: print(f"{ep+1}/{n_eps}", end=" ", flush=True)
    return pol

def evaluate(env, pol, qs, runs=3):
    res=[]
    for _ in range(runs):
        for q in qs:
            env.queries=[q]; s=env.new_query()
            while True:
                with torch.no_grad(): a=pol(torch.FloatTensor(s).unsqueeze(0)).argmax(1).item()
                s2,rw,done=env.step(a)
                if done: res.append(rw); break
                s=s2
    return res

def eval_fixed(env, qs, target_k):
    res=[]
    for q in qs:
        env.queries=[q]; env.new_query()
        for _ in range(target_k//env.dk): env.step(1)
        _,rw,_=env.step(0); res.append(rw)
    return res

# ── Run ──
tr_env = Env(train_q, N_FULL); te_env = Env(test_q, N_FULL)

print("Training BoR DQN...", end=" ", flush=True)
bor_pol = train_dqn(tr_env, 'bor')
print("done")
print("Training F1 DQN...", end=" ", flush=True)
f1_pol = train_dqn(tr_env, 'f1')
print("done\n")

br=evaluate(te_env, bor_pol, test_q)
fr=evaluate(te_env, f1_pol, test_q)
f10=eval_fixed(te_env, test_q, 10)
f20=eval_fixed(te_env, test_q, 20)
f50=eval_fixed(te_env, test_q, 50)

def row(res, label):
    k=np.mean([r['k'] for r in res]); f=np.mean([r['found'] for r in res])*100
    b=np.mean([r['bor'] for r in res if r['found']] or [0])
    print(f"  {label:<20} K={k:>5.1f}  Found={f:>5.1f}%  BoR={b:>5.2f}  eff={f/k:>.2f}%/doc")

print(f"{'─'*70}")
row(br, "BoR DQN"); row(fr, "F1 DQN")
row(f10, "Fixed K=10"); row(f20, "Fixed K=20"); row(f50, "Fixed K=50")
print(f"{'─'*70}")
print(f"  BoR K std: {np.std([r['k'] for r in br]):.2f}   F1 K std: {np.std([r['k'] for r in fr]):.2f}")

[INFO] Please confirm you agree to the MSMARCO data usage agreement found at <http://www.msmarco.org/dataset.aspx>
[INFO] If you have a local copy of https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz, you can symlink it here to avoid downloading it again: /root/.ir_datasets/downloads/31644046b18952c1386cd4564ba2ae69
[INFO] [starting] https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz
[INFO] [finished] https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz: [00:22] [1.06GB] [47.9MB/s]


Loading documents (50K target)...


[INFO] [starting] fixing encoding
[INFO] [finished] fixing encoding: [14:41] [3.06GB] [3.47MB/s]


  scanned 2M, stored 50027
  scanned 4M, stored 50071
  scanned 6M, stored 50113
  scanned 8M, stored 51019
Local index: 51025 docs

Building BM25 index... done
Computing BM25 rankings...
  100/500
  200/500
  300/500
  400/500
  500/500

500 queries, local index 51025, BoR against N=8,841,823
  K=  5: found= 69.4%  BoR_ceil=20.7 bits
  K= 10: found= 74.2%  BoR_ceil=19.7 bits
  K= 20: found= 77.0%  BoR_ceil=18.7 bits
  K= 50: found= 82.2%  BoR_ceil=17.4 bits
Train: 350, Test: 150

Training BoR DQN... 5000/20000 10000/20000 15000/20000 20000/20000 done
Training F1 DQN... 5000/20000 10000/20000 15000/20000 20000/20000 done

──────────────────────────────────────────────────────────────────────
  BoR DQN              K= 24.0  Found= 82.7%  BoR=20.28  eff=3.44%/doc
  F1 DQN               K= 16.8  Found= 78.7%  BoR=20.46  eff=4.69%/doc
  Fixed K=10           K= 10.0  Found= 73.3%  BoR=19.69  eff=7.33%/doc
  Fixed K=20           K= 20.0  Found= 75.3%  BoR=18.70  eff=3.77%/doc
  Fixed K=50   

In [ ]:
# Single Colab cell: MetaTool single-tool benchmark + BM25 scorer + adaptive STOP/CONTINUE DQN
# This is self-contained and CPU-friendly. It does NOT call an LLM; it benchmarks adaptive tool-presentation depth.

import sys, subprocess, pkgutil, io, os, re, json, math, random, time
from collections import deque

def ensure(import_name, pip_name=None):
    if pkgutil.find_loader(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

ensure("requests")
ensure("pandas")
ensure("numpy")
ensure("sklearn", "scikit-learn")
ensure("rank_bm25", "rank-bm25")
ensure("torch")

import requests
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from rank_bm25 import BM25Okapi
import torch
import torch.nn as nn
import torch.optim as optim

# -----------------------------
# Config
# -----------------------------
SEED = 42
MAX_QUERIES = 2000       # cap for speed; raise to use more of MetaTool
MAX_PER_TOOL = 20        # balance queries across tools
CANDIDATE_N = 100        # per-query candidate registry size; set to None to use full registry
HARD_NEG = 24            # number of BM25 hard distractors inside candidate set
TRAIN_EPISODES = 12000   # 8k-20k is reasonable on Colab CPU
BATCH_SIZE = 128
REPLAY_SIZE = 50000
LR = 1e-3
GAMMA = 1.0             # no extra depth penalty beyond the reward itself
TARGET_UPDATE_EVERY = 500
PRINT_EVERY = 3000

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# Data URLs (MetaTool official repo)
# -----------------------------
CSV_URL = "https://raw.githubusercontent.com/HowieHwong/MetaTool/master/dataset/data/all_clean_data.csv"
TOOLS_URL = "https://raw.githubusercontent.com/HowieHwong/MetaTool/master/dataset/plugin_des.json"

# -----------------------------
# Helpers
# -----------------------------
def normalize_name(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).lower())

def tokenize(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())

def pct(x):
    return f"{100*x:.1f}%"

def bor_reward(success, k, N):
    if not success:
        return 0.0
    p_rand = min(1.0, max(k / N, 1e-12))   # exact for R_q = 1, single relevant tool
    return max(0.0, -math.log2(p_rand))

def f1_reward(success, k):
    # For a single relevant tool:
    # precision = 1/k if found else 0
    # recall = 1 if found else 0
    # F1 = 2/(k+1) if found else 0
    return 0.0 if not success else (2.0 / (k + 1.0))

# -----------------------------
# Download + parse MetaTool
# -----------------------------
print("Downloading MetaTool...")
csv_text = requests.get(CSV_URL, timeout=120).text
tools_text = requests.get(TOOLS_URL, timeout=120).text

df = pd.read_csv(io.StringIO(csv_text), engine="python")
df.columns = [c.strip() for c in df.columns]

query_col = next((c for c in df.columns if c.lower() == "query"), None)
tool_col = next((c for c in df.columns if c.lower() == "tool"), None)
if query_col is None or tool_col is None:
    raise ValueError(f"Could not find Query/Tool columns. Found columns: {list(df.columns)}")

tool_desc_raw = json.loads(tools_text)

# Normalize tool registry
registry = {}
for tool_name, desc in tool_desc_raw.items():
    key = normalize_name(tool_name)
    if key and key not in registry:
        registry[key] = {
            "name": tool_name,
            "description": " ".join(str(desc).split())
        }

df = df[[query_col, tool_col]].copy()
df[query_col] = df[query_col].astype(str).str.strip()
df[tool_col] = df[tool_col].astype(str).str.strip()
df["tool_norm"] = df[tool_col].map(normalize_name)

df = df[df[query_col].str.len() > 0]
df = df[df["tool_norm"].isin(registry)]
df = df.drop_duplicates(subset=[query_col, "tool_norm"]).reset_index(drop=True)

# Balance queries across tools a bit so a few popular tools do not dominate
balanced_parts = []
for _, g in df.groupby("tool_norm"):
    n = min(len(g), MAX_PER_TOOL)
    balanced_parts.append(g.sample(n=n, random_state=SEED))
data = pd.concat(balanced_parts, ignore_index=True)

if len(data) > MAX_QUERIES:
    data = data.sample(n=MAX_QUERIES, random_state=SEED)

data = data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

tool_keys = sorted(registry.keys())
tool_names = [registry[k]["name"] for k in tool_keys]
tool_descs = [registry[k]["description"] for k in tool_keys]
tool_to_idx = {k: i for i, k in enumerate(tool_keys)}

FULL_REGISTRY_N = len(tool_keys)
if CANDIDATE_N is None:
    CANDIDATE_N = FULL_REGISTRY_N
CANDIDATE_N = min(CANDIDATE_N, FULL_REGISTRY_N)
HARD_NEG = min(HARD_NEG, CANDIDATE_N - 1)

print(f"Loaded MetaTool single-tool data: {len(data)} queries")
print(f"Full tool registry: {FULL_REGISTRY_N} tools")
print(f"Per-query candidate set: {CANDIDATE_N} tools ({HARD_NEG} hard BM25 distractors + random fill)")

# -----------------------------
# BM25 scorer over tool descriptions
# -----------------------------
bm25 = BM25Okapi([tokenize(d) for d in tool_descs])

def build_instance(row, rng):
    q = row[query_col]
    gold_key = row["tool_norm"]
    gold_idx = tool_to_idx[gold_key]

    scores_all = np.asarray(bm25.get_scores(tokenize(q)), dtype=np.float32)

    # Build per-query candidate set:
    #   1 gold tool + top hard negatives from BM25 + random fill from remainder
    ranked_global = np.argsort(-scores_all)
    hard = [i for i in ranked_global if i != gold_idx][:HARD_NEG]

    hard_set = set(hard + [gold_idx])
    remaining = [i for i in range(FULL_REGISTRY_N) if i not in hard_set]
    fill_n = CANDIDATE_N - 1 - len(hard)
    fill = rng.sample(remaining, k=fill_n) if fill_n > 0 else []

    cand = [gold_idx] + hard + fill
    cand_scores = np.asarray([scores_all[i] for i in cand], dtype=np.float32)

    order = np.argsort(-cand_scores)
    ranked_tool_ids = [cand[i] for i in order]
    ranked_scores = cand_scores[order]
    gold_rank = ranked_tool_ids.index(gold_idx) + 1

    return {
        "query": q,
        "gold_tool": registry[gold_key]["name"],
        "gold_rank": int(gold_rank),
        "ranked_tool_ids": ranked_tool_ids,
        "scores": ranked_scores,
        "N": int(len(ranked_tool_ids)),
    }

rng_build = random.Random(SEED)
instances = [build_instance(row, rng_build) for _, row in data.iterrows()]

# Quick sanity print
print("\nStatic BM25 baselines:")
for K in [1, 3, 5, 10, 20, 50, 100]:
    if K > CANDIDATE_N:
        continue
    found = np.mean([inst["gold_rank"] <= K for inst in instances])
    p_rand = K / CANDIDATE_N
    bor_ceil = max(0.0, -math.log2(p_rand))
    print(f"  K={K:>3}: found={pct(found):>6}  P_rand={p_rand:>6.4f}  BoR_ceil={bor_ceil:>4.1f} bits")

hard_examples = [inst for inst in instances if inst["gold_rank"] > 3]
ex = random.choice(hard_examples if hard_examples else instances)
print("\nExample query:")
print(f"  {ex['query']}")
print(f"  Gold tool: {ex['gold_tool']}")
print("  BM25 top-5:")
for i, tid in enumerate(ex["ranked_tool_ids"][:5], start=1):
    mark = " ✓" if tool_names[tid] == ex["gold_tool"] else ""
    print(f"    {tool_names[tid][:34]:34s} score={ex['scores'][i-1]:6.2f}{mark}")

# -----------------------------
# Train/test split
# -----------------------------
train_insts, test_insts = train_test_split(instances, test_size=0.30, random_state=SEED)
print(f"\nTrain: {len(train_insts)}, Test: {len(test_insts)}")

# -----------------------------
# MDP state
# -----------------------------
def state_vec(inst, k):
    # State after showing top-k tools, deciding STOP vs CONTINUE
    # k is 1-based and always in [1, N]
    s = inst["scores"]
    N = inst["N"]
    idx = min(k - 1, N - 1)

    cur = float(s[idx])
    nxt = float(s[idx + 1]) if idx + 1 < N else float(s[idx])
    first = float(s[0])
    mean = float(s.mean())
    std = float(s.std() + 1e-6)
    gap = cur - nxt if idx + 1 < N else 0.0

    feats = np.array([
        k / N,
        math.log2(k + 1) / math.log2(N + 1),
        cur,
        nxt,
        gap,
        (cur - mean) / std,
        cur / (abs(first) + 1e-6),
    ], dtype=np.float32)
    return feats

STATE_DIM = len(state_vec(train_insts[0], 1))

# -----------------------------
# DQN
# -----------------------------
class QNet(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 2)   # 0=STOP, 1=CONTINUE
        )
    def forward(self, x):
        return self.net(x)

def train_dqn(train_instances, reward_name="bor"):
    reward_fn = bor_reward if reward_name == "bor" else f1_reward

    net = QNet(STATE_DIM).to(device)
    target = QNet(STATE_DIM).to(device)
    target.load_state_dict(net.state_dict())

    opt = optim.Adam(net.parameters(), lr=LR)
    replay = deque(maxlen=REPLAY_SIZE)

    epsilon_start, epsilon_end = 1.0, 0.05
    global_step = 0

    def greedy_action(state_np):
        with torch.no_grad():
            q = net(torch.tensor(state_np, dtype=torch.float32, device=device).unsqueeze(0))
            return int(q.argmax(dim=1).item())

    for ep in range(1, TRAIN_EPISODES + 1):
        inst = random.choice(train_instances)
        N = inst["N"]
        k = 1
        done = False

        epsilon = epsilon_end + (epsilon_start - epsilon_end) * max(0.0, 1.0 - ep / TRAIN_EPISODES)

        while not done:
            s = state_vec(inst, k)

            if random.random() < epsilon:
                a = random.randint(0, 1)
            else:
                a = greedy_action(s)

            # Cannot continue past N
            if k >= N:
                a = 0

            if a == 0:
                success = inst["gold_rank"] <= k
                r = reward_fn(success, k, inst["N"]) if reward_name == "bor" else reward_fn(success, k)
                replay.append((s, a, float(r), None, 1.0))
                done = True
            else:
                k2 = k + 1
                if k2 >= N:
                    success = inst["gold_rank"] <= N
                    r = reward_fn(success, N, inst["N"]) if reward_name == "bor" else reward_fn(success, N)
                    replay.append((s, a, float(r), None, 1.0))
                    done = True
                else:
                    s2 = state_vec(inst, k2)
                    replay.append((s, a, 0.0, s2, 0.0))
                    k = k2

            global_step += 1

            if len(replay) >= BATCH_SIZE:
                batch = random.sample(replay, BATCH_SIZE)

                states = torch.tensor(np.stack([b[0] for b in batch]), dtype=torch.float32, device=device)
                actions = torch.tensor([b[1] for b in batch], dtype=torch.long, device=device).unsqueeze(1)
                rewards = torch.tensor([b[2] for b in batch], dtype=torch.float32, device=device)
                dones = torch.tensor([b[4] for b in batch], dtype=torch.float32, device=device)

                nonfinal_mask = torch.tensor([b[3] is not None for b in batch], dtype=torch.bool, device=device)
                next_q = torch.zeros(BATCH_SIZE, dtype=torch.float32, device=device)

                if nonfinal_mask.any():
                    next_states = torch.tensor(
                        np.stack([b[3] for b in batch if b[3] is not None]),
                        dtype=torch.float32, device=device
                    )
                    with torch.no_grad():
                        next_q[nonfinal_mask] = target(next_states).max(dim=1).values

                q_values = net(states).gather(1, actions).squeeze(1)
                target_values = rewards + (1.0 - dones) * GAMMA * next_q

                loss = nn.SmoothL1Loss()(q_values, target_values)
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(), 5.0)
                opt.step()

            if global_step % TARGET_UPDATE_EVERY == 0:
                target.load_state_dict(net.state_dict())

        if ep % PRINT_EVERY == 0:
            print(f"Training {reward_name.upper()} DQN... {ep}/{TRAIN_EPISODES}")

    return net

# -----------------------------
# Evaluation
# -----------------------------
def rollout_model(inst, model):
    k = 1
    while True:
        s = state_vec(inst, k)
        with torch.no_grad():
            a = int(model(torch.tensor(s, dtype=torch.float32, device=device).unsqueeze(0)).argmax(dim=1).item())
        if a == 0 or k >= inst["N"]:
            break
        k += 1

    success = inst["gold_rank"] <= k
    return {
        "k": k,
        "success": success,
        "bor": bor_reward(success, k, inst["N"]),
        "f1": f1_reward(success, k),
    }

def eval_fixed(instances, K):
    rows = []
    for inst in instances:
        k = min(K, inst["N"])
        success = inst["gold_rank"] <= k
        rows.append({
            "k": k,
            "success": success,
            "bor": bor_reward(success, k, inst["N"]),
            "f1": f1_reward(success, k),
        })
    return rows

def eval_model(instances, model):
    return [rollout_model(inst, model) for inst in instances]

def summarize(name, rows):
    ks = np.array([r["k"] for r in rows], dtype=float)
    found = np.mean([r["success"] for r in rows])
    bor = np.mean([r["bor"] for r in rows])
    f1m = np.mean([r["f1"] for r in rows])
    print(f"  {name:18s} K={ks.mean():6.1f}  Found={pct(found):>6}  BoR={bor:5.2f}  F1={f1m:5.3f}")
    return ks

# -----------------------------
# Train
# -----------------------------
t0 = time.time()
bor_model = train_dqn(train_insts, reward_name="bor")
f1_model = train_dqn(train_insts, reward_name="f1")
print(f"\nTraining finished in {(time.time()-t0):.1f}s on {device}")

# -----------------------------
# Test results
# -----------------------------
bor_rows = eval_model(test_insts, bor_model)
f1_rows = eval_model(test_insts, f1_model)

fixed_rows = {}
for K in [1, 3, 5, 10, 20, 50, CANDIDATE_N]:
    if K <= CANDIDATE_N:
        fixed_rows[K] = eval_fixed(test_insts, K)

print(f"\nMETA TOOL SELECTION: {FULL_REGISTRY_N} total tools, {CANDIDATE_N}-tool candidate set, single-tool queries (R_q=1)")
print("─" * 72)
bor_ks = summarize("BoR DQN", bor_rows)
f1_ks = summarize("F1 DQN", f1_rows)
for K in [1, 3, 5, 10, 20, 50]:
    if K in fixed_rows:
        summarize(f"Fixed K={K}", fixed_rows[K])
summarize(f"Fixed K={CANDIDATE_N} (all)", fixed_rows[CANDIDATE_N])
print("─" * 72)
print(f"  BoR K std: {np.std(bor_ks):.2f}   F1 K std: {np.std(f1_ks):.2f}")

# Optional: inspect some per-query decisions
print("\nSample BoR-policy decisions on test queries:")
for inst, row in list(zip(test_insts[:5], bor_rows[:5])):
    shown = row["k"]
    top_tools = [tool_names[i] for i in inst["ranked_tool_ids"][:min(shown, 5)]]
    print(f"- Query: {inst['query'][:100]}")
    print(f"  Gold: {inst['gold_tool']} | chosen K={shown} | success={row['success']} | gold_rank={inst['gold_rank']}")
    print(f"  Shown top tools: {top_tools}")

/tmp/ipykernel_26295/1527889703.py:8: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  if pkgutil.find_loader(import_name) is None:


Loaded MetaTool single-tool data: 2000 queries
Full tool registry: 199 tools
Per-query candidate set: 100 tools (24 hard BM25 distractors + random fill)

Static BM25 baselines:
  K=  1: found= 35.0%  P_rand=0.0100  BoR_ceil= 6.6 bits
  K=  3: found= 47.9%  P_rand=0.0300  BoR_ceil= 5.1 bits
  K=  5: found= 53.4%  P_rand=0.0500  BoR_ceil= 4.3 bits
  K= 10: found= 59.2%  P_rand=0.1000  BoR_ceil= 3.3 bits
  K= 20: found= 66.8%  P_rand=0.2000  BoR_ceil= 2.3 bits
  K= 50: found= 84.2%  P_rand=0.5000  BoR_ceil= 1.0 bits
  K=100: found=100.0%  P_rand=1.0000  BoR_ceil= 0.0 bits

Example query:
  I don't understand the concept of time dilation in Einstein's theory of relativity. Can you simplify it for me?
  Gold tool: reflect_notes
  BM25 top-5:
    TicTacToe                          score=  9.25
    locator                            score=  8.47
    CribbageScorer                     score=  8.32
    mbti                               score=  8.21
    Bohita                             score=

In [ ]:
"""
BoR-as-Reward: MetaTool + Embeddings (step cost, consistent with retrieval).
Same reward structure as SciFact/NFCorpus/MARCO: pure BoR at STOP, -step_cost per CONTINUE.
pip install requests pandas rank_bm25 torch sentence-transformers
"""
import sys, subprocess, pkgutil, io, re, json, math, random, time, numpy as np, pandas as pd
from collections import deque

def ensure(name, pip=None):
    if pkgutil.find_loader(name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip or name])
ensure("requests"); ensure("pandas"); ensure("numpy"); ensure("sklearn", "scikit-learn")
ensure("rank_bm25", "rank-bm25"); ensure("torch"); ensure("sentence_transformers", "sentence-transformers")

import requests, torch, torch.nn as nn, torch.optim as optim
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer

# ── Config ──
SEED = 42
MAX_QUERIES = 2000
MAX_PER_TOOL = 20
CANDIDATE_N = 100
HARD_NEG = 24
TRAIN_EPS = 15000
BATCH = 128
REPLAY = 50000
LR = 1e-3
GAMMA = 0.95          # same as retrieval experiments
STEP_COST = 0.01      # same as retrieval experiments
TARGET_EVERY = 500

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# ── Reward functions (pure — no failure penalty, no floor) ──
def bor_reward(success, k, N):
    p_rand = max(k / N, 1e-12)
    return -math.log2(p_rand) if success else 0.0

def f1_reward(success, k):
    return (2.0 / (k + 1.0)) if success else 0.0

# ── Download MetaTool ──
print("Downloading MetaTool...", flush=True)
csv_text = requests.get("https://raw.githubusercontent.com/HowieHwong/MetaTool/master/dataset/data/all_clean_data.csv", timeout=120).text
tools_text = requests.get("https://raw.githubusercontent.com/HowieHwong/MetaTool/master/dataset/plugin_des.json", timeout=120).text

normalize = lambda x: re.sub(r"[^a-z0-9]+", "", str(x).lower())

df = pd.read_csv(io.StringIO(csv_text), engine="python")
df.columns = [c.strip() for c in df.columns]
qcol = next(c for c in df.columns if c.lower() == "query")
tcol = next(c for c in df.columns if c.lower() == "tool")
tool_desc_raw = json.loads(tools_text)

registry = {}
for name, desc in tool_desc_raw.items():
    key = normalize(name)
    if key and key not in registry:
        registry[key] = {"name": name, "description": " ".join(str(desc).split())}

df = df[[qcol, tcol]].copy()
df[qcol] = df[qcol].astype(str).str.strip()
df[tcol] = df[tcol].astype(str).str.strip()
df["norm"] = df[tcol].map(normalize)
df = df[df[qcol].str.len() > 0]
df = df[df["norm"].isin(registry)]
df = df.drop_duplicates(subset=[qcol, "norm"]).reset_index(drop=True)

parts = [g.sample(n=min(len(g), MAX_PER_TOOL), random_state=SEED) for _, g in df.groupby("norm")]
data = pd.concat(parts, ignore_index=True)
if len(data) > MAX_QUERIES: data = data.sample(n=MAX_QUERIES, random_state=SEED)
data = data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

tool_keys = sorted(registry.keys())
tool_names = [registry[k]["name"] for k in tool_keys]
tool_descs = [registry[k]["description"] for k in tool_keys]
tool_to_idx = {k: i for i, k in enumerate(tool_keys)}
N_FULL = len(tool_keys)

print(f"MetaTool: {len(data)} queries, {N_FULL} tools, {CANDIDATE_N}-tool candidates")

# ── Embedding scorer ──
print("Loading sentence-transformer...", flush=True)
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
tool_embs = encoder.encode(tool_descs, batch_size=64, show_progress_bar=False, normalize_embeddings=True)
query_texts = data[qcol].tolist()
print("Encoding queries...", flush=True)
query_embs = encoder.encode(query_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
score_matrix = query_embs @ tool_embs.T

# ── Build instances ──
def build_instance(i, row, rng):
    gold_key = row["norm"]
    gold_idx = tool_to_idx[gold_key]
    scores_all = score_matrix[i]
    ranked_global = np.argsort(-scores_all)
    hard = [j for j in ranked_global if j != gold_idx][:HARD_NEG]
    hard_set = set(hard + [gold_idx])
    remaining = [j for j in range(N_FULL) if j not in hard_set]
    fill = rng.sample(remaining, k=max(0, CANDIDATE_N - 1 - len(hard)))
    cand = [gold_idx] + hard + fill
    cand_scores = np.array([scores_all[j] for j in cand], dtype=np.float32)
    order = np.argsort(-cand_scores)
    ranked_ids = [cand[j] for j in order]
    ranked_sc = cand_scores[order]
    gold_rank = ranked_ids.index(gold_idx) + 1
    return {"gold_rank": gold_rank, "ranked_tool_ids": ranked_ids,
            "scores": ranked_sc, "N": len(ranked_ids),
            "query": row[qcol], "gold_tool": registry[gold_key]["name"]}

rng = random.Random(SEED)
instances = [build_instance(i, row, rng) for i, (_, row) in enumerate(data.iterrows())]

print("\nStatic baselines:")
for K in [1, 3, 5, 10, 20, 50, 100]:
    if K > CANDIDATE_N: break
    found = np.mean([inst["gold_rank"] <= K for inst in instances])
    p_rand = K / CANDIDATE_N
    print(f"  K={K:>3}: found={100*found:>5.1f}%  P_rand={p_rand:.4f}  BoR_ceil={max(0,-math.log2(p_rand)):.1f} bits")

train_insts, test_insts = train_test_split(instances, test_size=0.30, random_state=SEED)
print(f"\nTrain: {len(train_insts)}, Test: {len(test_insts)}")
print(f"Step cost: {STEP_COST}, Gamma: {GAMMA} (same as retrieval experiments)\n")

# ── State (same as GPT's MetaTool cell) ──
def state_vec(inst, k):
    s = inst["scores"]; N = inst["N"]; idx = min(k-1, N-1)
    cur = float(s[idx])
    nxt = float(s[idx+1]) if idx+1 < N else float(s[idx])
    first = float(s[0]); mean = float(s.mean()); std = float(s.std() + 1e-6)
    gap = cur - nxt if idx+1 < N else 0.0
    return np.array([k/N, math.log2(k+1)/math.log2(N+1), cur, nxt, gap,
                     (cur-mean)/std, cur/(abs(first)+1e-6)], dtype=np.float32)

# ── DQN ──
class QNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(7,64), nn.ReLU(), nn.Linear(64,64), nn.ReLU(), nn.Linear(64,2))
    def forward(self, x): return self.net(x)

def train_dqn(train_insts, reward_name="bor"):
    rfn = bor_reward if reward_name == "bor" else f1_reward
    net = QNet().to(device); tgt = QNet().to(device); tgt.load_state_dict(net.state_dict())
    opt = optim.Adam(net.parameters(), lr=LR)
    replay = deque(maxlen=REPLAY)
    eps_start, eps_end = 1.0, 0.05; step = 0

    for ep in range(1, TRAIN_EPS+1):
        inst = random.choice(train_insts); N = inst["N"]; k = 1
        eps = eps_end + (eps_start - eps_end) * max(0, 1 - ep/TRAIN_EPS)
        done = False
        while not done:
            sv = state_vec(inst, k)
            if random.random() < eps: a = random.randint(0,1)
            else:
                with torch.no_grad():
                    a = int(net(torch.tensor(sv, device=device).unsqueeze(0)).argmax(1).item())
            if k >= N: a = 0

            if a == 0:  # STOP — pure reward, no penalty
                success = inst["gold_rank"] <= k
                r = rfn(success, k, inst["N"]) if reward_name == "bor" else rfn(success, k)
                replay.append((sv, a, float(r), None, 1.0)); done = True
            else:  # CONTINUE — pay step cost
                k2 = k + 1
                if k2 >= N:
                    success = inst["gold_rank"] <= N
                    r = rfn(success, N, inst["N"]) if reward_name == "bor" else rfn(success, N)
                    replay.append((sv, a, float(r), None, 1.0)); done = True
                else:
                    sv2 = state_vec(inst, k2)
                    replay.append((sv, a, -STEP_COST, sv2, 0.0))
                    k = k2

            step += 1
            if len(replay) >= BATCH:
                batch = random.sample(replay, BATCH)
                states = torch.tensor(np.stack([b[0] for b in batch]), dtype=torch.float32, device=device)
                actions = torch.tensor([b[1] for b in batch], dtype=torch.long, device=device).unsqueeze(1)
                rewards = torch.tensor([b[2] for b in batch], dtype=torch.float32, device=device)
                dones = torch.tensor([b[4] for b in batch], dtype=torch.float32, device=device)
                nf = torch.tensor([b[3] is not None for b in batch], dtype=torch.bool, device=device)
                nq = torch.zeros(BATCH, dtype=torch.float32, device=device)
                if nf.any():
                    ns = torch.tensor(np.stack([b[3] for b in batch if b[3] is not None]), dtype=torch.float32, device=device)
                    with torch.no_grad(): nq[nf] = tgt(ns).max(1).values
                qv = net(states).gather(1, actions).squeeze(1)
                tv = rewards + (1-dones) * GAMMA * nq
                loss = nn.SmoothL1Loss()(qv, tv)
                opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 5.0); opt.step()
            if step % TARGET_EVERY == 0: tgt.load_state_dict(net.state_dict())
        if ep % 3000 == 0: print(f"  {ep}/{TRAIN_EPS}", end="", flush=True)
    return net

# ── Eval (pure BoR, no penalty) ──
def rollout(inst, model):
    k = 1
    while True:
        sv = state_vec(inst, k)
        with torch.no_grad():
            a = int(model(torch.tensor(sv, device=device).unsqueeze(0)).argmax(1).item())
        if a == 0 or k >= inst["N"]: break
        k += 1
    success = inst["gold_rank"] <= k
    p_rand = max(k / inst["N"], 1e-12)
    return {"k": k, "success": success,
            "bor": max(0, -math.log2(p_rand)) if success else 0.0,
            "f1": (2.0/(k+1)) if success else 0.0}

def eval_fixed(insts, K):
    rows = []
    for inst in insts:
        k = min(K, inst["N"]); success = inst["gold_rank"] <= k
        p_rand = max(k / inst["N"], 1e-12)
        rows.append({"k": k, "success": success,
                      "bor": max(0, -math.log2(p_rand)) if success else 0.0,
                      "f1": (2.0/(k+1)) if success else 0.0})
    return rows

def row(rows, label):
    ks = np.array([r["k"] for r in rows], dtype=float)
    f = np.mean([r["success"] for r in rows])
    b = np.mean([r["bor"] for r in rows])
    f1 = np.mean([r["f1"] for r in rows])
    print(f"  {label:18s} K={ks.mean():>6.1f}  Found={100*f:>5.1f}%  BoR={b:>5.2f}  F1={f1:>5.3f}")
    return ks

# ── Run ──
t0 = time.time()
print("Training BoR DQN...", end="", flush=True)
bor_model = train_dqn(train_insts, "bor"); print(" done")
print("Training F1 DQN...", end="", flush=True)
f1_model = train_dqn(train_insts, "f1"); print(" done")
print(f"Training: {time.time()-t0:.0f}s on {device}\n")

br = [rollout(i, bor_model) for i in test_insts]
fr = [rollout(i, f1_model) for i in test_insts]

print(f"  METATOOL + EMBEDDINGS: {N_FULL} tools, {len(test_insts)} test, step_cost={STEP_COST}, gamma={GAMMA}")
print("─" * 72)
bk = row(br, "BoR DQN"); fk = row(fr, "F1 DQN")
for K in [1, 3, 5, 10, 20, 50]:
    row(eval_fixed(test_insts, K), f"Fixed K={K}")
row(eval_fixed(test_insts, CANDIDATE_N), f"Fixed K={CANDIDATE_N} (all)")
print("─" * 72)
print(f"  BoR K std: {np.std(bk):.2f}   F1 K std: {np.std(fk):.2f}")

# Sample decisions
print("\nSample BoR decisions:")
for inst, r in list(zip(test_insts[:8], br[:8])):
    print(f"  K={r['k']:>3} {'✓' if r['success'] else '✗'} rank={inst['gold_rank']:>2} | {inst['query'][:70]}")

/tmp/ipykernel_26295/1815228932.py:10: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  if pkgutil.find_loader(name) is None:


MetaTool: 2000 queries, 199 tools, 100-tool candidates
Loading sentence-transformer...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding queries...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]


Static baselines:
  K=  1: found= 60.8%  P_rand=0.0100  BoR_ceil=6.6 bits
  K=  3: found= 75.1%  P_rand=0.0300  BoR_ceil=5.1 bits
  K=  5: found= 79.1%  P_rand=0.0500  BoR_ceil=4.3 bits
  K= 10: found= 84.0%  P_rand=0.1000  BoR_ceil=3.3 bits
  K= 20: found= 88.9%  P_rand=0.2000  BoR_ceil=2.3 bits
  K= 50: found= 96.3%  P_rand=0.5000  BoR_ceil=1.0 bits
  K=100: found=100.0%  P_rand=1.0000  BoR_ceil=0.0 bits

Train: 1400, Test: 600
Step cost: 0.01, Gamma: 0.95 (same as retrieval experiments)

Training BoR DQN...  3000/15000  6000/15000  9000/15000  12000/15000  15000/15000 done
Training F1 DQN...  3000/15000  6000/15000  9000/15000  12000/15000  15000/15000 done
Training: 209s on cpu

  METATOOL + EMBEDDINGS: 199 tools, 600 test, step_cost=0.01, gamma=0.95
────────────────────────────────────────────────────────────────────────
  BoR DQN            K=   2.3  Found= 73.3%  BoR= 4.44  F1=0.605
  F1 DQN             K=   3.0  Found= 69.0%  BoR= 4.24  F1=0.604
  Fixed K=1          K=   1.0  

In [ ]:
# Self-contained Colab cell:
# MetaTool + sentence-transformer scorer + adaptive STOP/CONTINUE DQN
# Story-focused version:
#   1) varies candidate-set size N in {20, 50, 100}
#   2) uses HARD distractors only (no random fill)
#   3) reports difficulty-bucket behavior so adaptivity is visible
#
# Important experimental note:
#   The gold tool is ALWAYS included in the candidate set.
#   This isolates presentation-depth control rather than full-registry recall.
#
# Runtime:
#   CPU: a few minutes
#   GPU: faster
#
# Adjust MAX_QUERIES / TRAIN_EPISODES if needed.

import sys, subprocess, importlib.util, io, re, json, math, random, time
from collections import deque, defaultdict

def ensure(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

ensure("requests")
ensure("pandas")
ensure("numpy")
ensure("sklearn", "scikit-learn")
ensure("torch")
ensure("sentence_transformers", "sentence-transformers")

import requests
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from sentence_transformers import SentenceTransformer

try:
    from IPython.display import display
except:
    display = print

# -----------------------------
# Config
# -----------------------------
SEED = 42
MAX_QUERIES = 1200
MAX_PER_TOOL = 20
CANDIDATE_SIZES = [20, 50, 100]   # vary registry size
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

TRAIN_EPISODES = 5000
BATCH_SIZE = 128
REPLAY_SIZE = 50000
LR = 1e-3
GAMMA = 0.95
STEP_COST = 0.01
TARGET_UPDATE_EVERY = 500
PRINT_EVERY = 2500

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# Data URLs (MetaTool official repo)
# -----------------------------
CSV_URL = "https://raw.githubusercontent.com/HowieHwong/MetaTool/master/dataset/data/all_clean_data.csv"
TOOLS_URL = "https://raw.githubusercontent.com/HowieHwong/MetaTool/master/dataset/plugin_des.json"

# -----------------------------
# Helpers
# -----------------------------
def normalize_name(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).lower())

def pct(x):
    return f"{100*x:.1f}%"

def bor_reward(success, k, N):
    # exact for single-tool queries (R_q = 1): P_rand = k / N
    if not success:
        return 0.0
    p_rand = min(1.0, max(k / N, 1e-12))
    return max(0.0, -math.log2(p_rand))

def f1_reward(success, k):
    # single relevant tool:
    # F1 = 2/(k+1) if found else 0
    return 0.0 if not success else (2.0 / (k + 1.0))

def summarize_rows(name, rows):
    ks = np.array([r["k"] for r in rows], dtype=float)
    found = np.mean([r["success"] for r in rows]) if rows else 0.0
    bor = np.mean([r["bor"] for r in rows]) if rows else 0.0
    f1m = np.mean([r["f1"] for r in rows]) if rows else 0.0
    return {
        "policy": name,
        "K_mean": float(ks.mean()) if len(ks) else 0.0,
        "K_std": float(ks.std()) if len(ks) else 0.0,
        "Found": float(found),
        "BoR": float(bor),
        "F1": float(f1m),
        "n": len(rows),
    }

def difficulty_bucket(rank):
    if rank == 1:
        return "1"
    if 2 <= rank <= 3:
        return "2-3"
    if 4 <= rank <= 10:
        return "4-10"
    return "11+"

# -----------------------------
# Download + parse MetaTool
# -----------------------------
print("Downloading MetaTool...")
csv_text = requests.get(CSV_URL, timeout=120).text
tools_text = requests.get(TOOLS_URL, timeout=120).text

df = pd.read_csv(io.StringIO(csv_text), engine="python")
df.columns = [c.strip() for c in df.columns]

query_col = next((c for c in df.columns if c.lower() == "query"), None)
tool_col = next((c for c in df.columns if c.lower() == "tool"), None)
if query_col is None or tool_col is None:
    raise ValueError(f"Could not find Query/Tool columns. Found columns: {list(df.columns)}")

tool_desc_raw = json.loads(tools_text)

registry = {}
for tool_name, desc in tool_desc_raw.items():
    key = normalize_name(tool_name)
    if key and key not in registry:
        registry[key] = {
            "name": tool_name,
            "description": " ".join(str(desc).split())
        }

df = df[[query_col, tool_col]].copy()
df[query_col] = df[query_col].astype(str).str.strip()
df[tool_col] = df[tool_col].astype(str).str.strip()
df["tool_norm"] = df[tool_col].map(normalize_name)

df = df[df[query_col].str.len() > 0]
df = df[df["tool_norm"].isin(registry)]
df = df.drop_duplicates(subset=[query_col, "tool_norm"]).reset_index(drop=True)

# Balance across tools a bit
balanced_parts = []
for _, g in df.groupby("tool_norm"):
    n = min(len(g), MAX_PER_TOOL)
    balanced_parts.append(g.sample(n=n, random_state=SEED))
data = pd.concat(balanced_parts, ignore_index=True)

if len(data) > MAX_QUERIES:
    data = data.sample(n=MAX_QUERIES, random_state=SEED)

data = data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

tool_keys = sorted(registry.keys())
tool_names = [registry[k]["name"] for k in tool_keys]
tool_descs = [registry[k]["description"] for k in tool_keys]
tool_to_idx = {k: i for i, k in enumerate(tool_keys)}
FULL_REGISTRY_N = len(tool_keys)

print(f"MetaTool: {len(data)} queries, {FULL_REGISTRY_N} tools")
print("Candidate sets use HARD distractors only + guaranteed gold inclusion.")

# -----------------------------
# Sentence-transformer scorer
# -----------------------------
print(f"Loading sentence-transformer: {MODEL_NAME}")
st_model = SentenceTransformer(MODEL_NAME, device=device)

print("Encoding tool descriptions...")
tool_embs = st_model.encode(
    tool_descs,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Encoding queries...")
queries = data[query_col].tolist()
query_embs = st_model.encode(
    queries,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

# full score matrix: [num_queries, num_tools]
score_matrix = np.asarray(query_embs @ tool_embs.T, dtype=np.float32)

# -----------------------------
# Base instances over full registry
# -----------------------------
base_instances = []
for i, (_, row) in enumerate(data.iterrows()):
    gold_key = row["tool_norm"]
    gold_idx = tool_to_idx[gold_key]
    scores_all = score_matrix[i]
    ranked_global = np.argsort(-scores_all)
    gold_rank_full = int(np.where(ranked_global == gold_idx)[0][0] + 1)
    base_instances.append({
        "query": row[query_col],
        "gold_tool": registry[gold_key]["name"],
        "gold_idx": gold_idx,
        "scores_all": scores_all,
        "ranked_global": ranked_global,
        "gold_rank_full": gold_rank_full,
    })

# Example of full-registry scorer quality
print("\nFull-registry static baselines:")
for K in [1, 3, 5, 10, 20, 50, 100]:
    if K > FULL_REGISTRY_N:
        continue
    found = np.mean([inst["gold_rank_full"] <= K for inst in base_instances])
    p_rand = K / FULL_REGISTRY_N
    ceil = max(0.0, -math.log2(p_rand))
    print(f"  K={K:>3}: found={pct(found):>6}  P_rand={p_rand:>6.4f}  BoR_ceil={ceil:>4.1f} bits")

# -----------------------------
# Candidate-set construction
# -----------------------------
def make_candidate_instance(base_inst, N):
    """
    Hard-distractor candidate set:
      gold tool + top (N-1) highest-scoring negatives
    This makes the stopping problem genuinely harder than random-fill candidates.
    """
    gold_idx = base_inst["gold_idx"]
    ranked_global = base_inst["ranked_global"]
    hard_negatives = [tid for tid in ranked_global if tid != gold_idx][:N-1]
    cand = [gold_idx] + hard_negatives

    scores = np.asarray([base_inst["scores_all"][tid] for tid in cand], dtype=np.float32)
    order = np.argsort(-scores)
    ranked_tool_ids = [cand[j] for j in order]
    ranked_scores = scores[order]
    gold_rank = int(ranked_tool_ids.index(gold_idx) + 1)

    return {
        "query": base_inst["query"],
        "gold_tool": base_inst["gold_tool"],
        "gold_rank": gold_rank,
        "gold_rank_full": base_inst["gold_rank_full"],
        "ranked_tool_ids": ranked_tool_ids,
        "scores": ranked_scores,
        "N": N,
        "bucket": difficulty_bucket(gold_rank),
    }

# consistent split across all N
indices = np.arange(len(base_instances))
train_idx, test_idx = train_test_split(indices, test_size=0.30, random_state=SEED)
print(f"\nTrain: {len(train_idx)}, Test: {len(test_idx)}")
print(f"Step cost: {STEP_COST}, Gamma: {GAMMA} (same style as retrieval experiments)")

# -----------------------------
# RL state
# -----------------------------
def state_vec(inst, k):
    s = inst["scores"]
    N = inst["N"]
    idx = min(k - 1, N - 1)

    cur = float(s[idx])
    nxt = float(s[idx + 1]) if idx + 1 < N else float(s[idx])
    first = float(s[0])
    mean = float(s.mean())
    std = float(s.std() + 1e-6)
    gap = cur - nxt if idx + 1 < N else 0.0

    feats = np.array([
        k / N,
        math.log2(k + 1) / math.log2(N + 1),
        cur,
        nxt,
        gap,
        (cur - mean) / std,
        cur / (abs(first) + 1e-6),
    ], dtype=np.float32)
    return feats

# -----------------------------
# DQN
# -----------------------------
class QNet(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 2)   # 0=STOP, 1=CONTINUE
        )
    def forward(self, x):
        return self.net(x)

def train_dqn(train_instances, reward_name="bor"):
    reward_fn = bor_reward if reward_name == "bor" else f1_reward
    state_dim = len(state_vec(train_instances[0], 1))

    net = QNet(state_dim).to(device)
    target = QNet(state_dim).to(device)
    target.load_state_dict(net.state_dict())
    opt = optim.Adam(net.parameters(), lr=LR)
    replay = deque(maxlen=REPLAY_SIZE)

    epsilon_start, epsilon_end = 1.0, 0.05
    global_step = 0

    def greedy_action(state_np):
        with torch.no_grad():
            q = net(torch.tensor(state_np, dtype=torch.float32, device=device).unsqueeze(0))
            return int(q.argmax(dim=1).item())

    for ep in range(1, TRAIN_EPISODES + 1):
        inst = random.choice(train_instances)
        N = inst["N"]
        k = 1
        done = False
        epsilon = epsilon_end + (epsilon_start - epsilon_end) * max(0.0, 1.0 - ep / TRAIN_EPISODES)

        while not done:
            s = state_vec(inst, k)

            if random.random() < epsilon:
                a = random.randint(0, 1)
            else:
                a = greedy_action(s)

            if k >= N:
                a = 0

            if a == 0:  # STOP
                success = inst["gold_rank"] <= k
                terminal = reward_fn(success, k, inst["N"]) if reward_name == "bor" else reward_fn(success, k)
                r = float(terminal)
                replay.append((s, a, r, None, 1.0))
                done = True
            else:       # CONTINUE
                k2 = k + 1
                if k2 >= N:
                    # If continuing reaches the end, pay step cost then terminal reward at N
                    success = inst["gold_rank"] <= N
                    terminal = reward_fn(success, N, inst["N"]) if reward_name == "bor" else reward_fn(success, N)
                    r = float(-STEP_COST + terminal)
                    replay.append((s, a, r, None, 1.0))
                    done = True
                else:
                    s2 = state_vec(inst, k2)
                    r = float(-STEP_COST)
                    replay.append((s, a, r, s2, 0.0))
                    k = k2

            global_step += 1

            if len(replay) >= BATCH_SIZE:
                batch = random.sample(replay, BATCH_SIZE)

                states = torch.tensor(np.stack([b[0] for b in batch]), dtype=torch.float32, device=device)
                actions = torch.tensor([b[1] for b in batch], dtype=torch.long, device=device).unsqueeze(1)
                rewards = torch.tensor([b[2] for b in batch], dtype=torch.float32, device=device)
                dones = torch.tensor([b[4] for b in batch], dtype=torch.float32, device=device)

                nonfinal_mask = torch.tensor([b[3] is not None for b in batch], dtype=torch.bool, device=device)
                next_q = torch.zeros(BATCH_SIZE, dtype=torch.float32, device=device)

                if nonfinal_mask.any():
                    next_states = torch.tensor(
                        np.stack([b[3] for b in batch if b[3] is not None]),
                        dtype=torch.float32, device=device
                    )
                    with torch.no_grad():
                        next_q[nonfinal_mask] = target(next_states).max(dim=1).values

                q_values = net(states).gather(1, actions).squeeze(1)
                target_values = rewards + (1.0 - dones) * GAMMA * next_q

                loss = nn.SmoothL1Loss()(q_values, target_values)
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(), 5.0)
                opt.step()

            if global_step % TARGET_UPDATE_EVERY == 0:
                target.load_state_dict(net.state_dict())

        if ep % PRINT_EVERY == 0:
            print(f"Training {reward_name.upper()} DQN... {ep}/{TRAIN_EPISODES}")

    return net

# -----------------------------
# Evaluation
# -----------------------------
def rollout_model(inst, model):
    k = 1
    while True:
        s = state_vec(inst, k)
        with torch.no_grad():
            a = int(model(torch.tensor(s, dtype=torch.float32, device=device).unsqueeze(0)).argmax(dim=1).item())
        if a == 0 or k >= inst["N"]:
            break
        k += 1

    success = inst["gold_rank"] <= k
    return {
        "k": k,
        "success": success,
        "bor": bor_reward(success, k, inst["N"]),
        "f1": f1_reward(success, k),
        "bucket": inst["bucket"],
        "gold_rank": inst["gold_rank"],
    }

def eval_fixed(instances, K):
    rows = []
    for inst in instances:
        k = min(K, inst["N"])
        success = inst["gold_rank"] <= k
        rows.append({
            "k": k,
            "success": success,
            "bor": bor_reward(success, k, inst["N"]),
            "f1": f1_reward(success, k),
            "bucket": inst["bucket"],
            "gold_rank": inst["gold_rank"],
        })
    return rows

def eval_model(instances, model):
    return [rollout_model(inst, model) for inst in instances]

def choose_best_fixed_k(train_instances, ks):
    train_scores = []
    for K in ks:
        rows = eval_fixed(train_instances, K)
        train_scores.append((K, np.mean([r["bor"] for r in rows])))
    best_k, best_bor = max(train_scores, key=lambda x: x[1])
    return best_k, train_scores

def bucket_table(rows_dict):
    # rows_dict: policy_name -> rows
    all_rows = []
    bucket_order = ["1", "2-3", "4-10", "11+"]
    for bucket in bucket_order:
        row_out = {"bucket": bucket}
        present = False
        for name, rows in rows_dict.items():
            sub = [r for r in rows if r["bucket"] == bucket]
            if len(sub) > 0:
                present = True
                stats = summarize_rows(name, sub)
                row_out[f"{name}_n"] = stats["n"]
                row_out[f"{name}_K"] = round(stats["K_mean"], 2)
                row_out[f"{name}_Found"] = round(100 * stats["Found"], 1)
                row_out[f"{name}_BoR"] = round(stats["BoR"], 2)
            else:
                row_out[f"{name}_n"] = 0
                row_out[f"{name}_K"] = None
                row_out[f"{name}_Found"] = None
                row_out[f"{name}_BoR"] = None
        if present:
            all_rows.append(row_out)
    return pd.DataFrame(all_rows)

# -----------------------------
# Main experiment loop
# -----------------------------
overall_rows = []
bucket_tables = {}

for N in CANDIDATE_SIZES:
    print("\n" + "=" * 90)
    print(f"CANDIDATE SET SIZE N = {N} (hard distractors only)")
    print("=" * 90)

    instances_N = [make_candidate_instance(inst, N) for inst in base_instances]
    train_insts = [instances_N[i] for i in train_idx]
    test_insts  = [instances_N[i] for i in test_idx]

    # static baselines for this N
    print("Static baselines:")
    static_ks = [1, 3, 5, 10, 20, 50, N]
    static_ks = [k for k in static_ks if k <= N]
    for K in static_ks:
        found = np.mean([inst["gold_rank"] <= K for inst in test_insts])
        p_rand = K / N
        ceil = max(0.0, -math.log2(p_rand))
        print(f"  K={K:>3}: found={pct(found):>6}  P_rand={p_rand:>6.4f}  BoR_ceil={ceil:>4.1f} bits")

    # difficulty distribution
    bucket_counts = pd.Series([inst["bucket"] for inst in test_insts]).value_counts().reindex(["1","2-3","4-10","11+"], fill_value=0)
    print("Test difficulty buckets:", dict(bucket_counts))

    # choose best fixed-K baseline on TRAIN by BoR
    best_fixed_k, train_fixed_scores = choose_best_fixed_k(train_insts, static_ks)
    print(f"Best fixed K on TRAIN by BoR: K={best_fixed_k}")

    # train models
    t0 = time.time()
    bor_model = train_dqn(train_insts, reward_name="bor")
    f1_model  = train_dqn(train_insts, reward_name="f1")
    print(f"Training time for N={N}: {(time.time()-t0):.1f}s on {device}")

    # eval
    bor_rows = eval_model(test_insts, bor_model)
    f1_rows = eval_model(test_insts, f1_model)
    best_fixed_rows = eval_fixed(test_insts, best_fixed_k)
    fixed1_rows = eval_fixed(test_insts, 1)

    # overall summary
    summary_df = pd.DataFrame([
        summarize_rows("BoR DQN", bor_rows),
        summarize_rows("F1 DQN", f1_rows),
        summarize_rows(f"Fixed K={best_fixed_k}", best_fixed_rows),
        summarize_rows("Fixed K=1", fixed1_rows),
    ])
    summary_df["N"] = N
    overall_rows.extend(summary_df.to_dict("records"))

    print(f"\nOVERALL RESULTS @ N={N}")
    display(summary_df[["N","policy","n","K_mean","K_std","Found","BoR","F1"]].round(3))

    # difficulty buckets
    bt = bucket_table({
        "BoR": bor_rows,
        "F1": f1_rows,
        f"K{best_fixed_k}": best_fixed_rows,
    })
    bucket_tables[N] = bt

    print(f"\nDIFFICULTY BUCKETS @ N={N}")
    display(bt)

    # sample decisions
    print(f"\nSample BoR decisions @ N={N}:")
    sample = bor_rows[:6]
    for inst, row in zip(test_insts[:6], sample):
        mark = "✓" if row["success"] else "✗"
        print(f"  K={row['k']:>3} {mark} rank={inst['gold_rank']:>2} bucket={inst['bucket']:<4} | {inst['query'][:90]}")

# -----------------------------
# Cross-N summary
# -----------------------------
cross_df = pd.DataFrame(overall_rows)
cross_df["Found_pct"] = 100 * cross_df["Found"]

print("\n" + "#" * 90)
print("CROSS-N SUMMARY")
print("#" * 90)
display(
    cross_df[["N","policy","K_mean","K_std","Found_pct","BoR","F1"]]
    .sort_values(["N","policy"])
    .round(3)
)

# Small pivot for readability
pivot = cross_df.pivot_table(index="N", columns="policy", values=["K_mean","Found_pct","BoR"], aggfunc="first")
print("\nCompact cross-N pivot:")
display(pivot.round(3))

print("\nDone.")
print("Interpretation guide:")
print("- If BoR is doing what you want, it should stay high on BoR as N grows.")
print("- Difficulty buckets should show higher K on harder queries (4-10, 11+) than on easy ones (rank 1).")
print("- If K barely changes across buckets, the policy is mostly learning a near-constant stop rule.")

MetaTool: 1200 queries, 199 tools
Candidate sets use HARD distractors only + guaranteed gold inclusion.
Loading sentence-transformer: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding tool descriptions...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

...


Sample BoR decisions @ N=100:
  K=  1 ✓ rank= 1 bucket=1    | I want to make some changes to my FPL team's defense. Which defenders should I consider?
  K=  2 ✓ rank= 1 bucket=1    | What font does this company use?
  K=  2 ✓ rank= 1 bucket=1    | It's so frustrating to search for international job listings. I need your assistance in fi
  K=  1 ✓ rank= 1 bucket=1    | Can you search for any art pieces related to Asian culture from The Metropolitan Museum of
  K=  1 ✓ rank= 1 bucket=1    | Hey Chatbot, I have a blog and I want to monetize my outgoing traffic. Can you help me wit
  K=  1 ✓ rank= 1 bucket=1    | I just started playing this life simulator and I am feeling lost. Can you guide me?

##########################################################################################
CROSS-N SUMMARY
##########################################################################################


,N,policy,K_mean,K_std,Found_pct,BoR,F1
0,20,BoR DQN,1.589,0.759,70.556,2.751,0.609
1,20,F1 DQN,1.258,0.693,63.333,2.652,0.605
2,20,Fixed K=1,1.000,0.000,59.167,2.557,0.592
3,20,Fixed K=1,1.000,0.000,59.167,2.557,0.592
4,50,BoR DQN,2.631,2.566,74.722,3.655,0.583
5,50,F1 DQN,1.222,0.700,63.889,3.480,0.598
6,50,Fixed K=1,1.000,0.000,59.167,3.339,0.592
7,50,Fixed K=1,1.000,0.000,59.167,3.339,0.592
8,100,BoR DQN,2.142,1.396,72.778,4.364,0.581
9,100,F1 DQN,1.522,5.249,63.333,4.059,0.588



Compact cross-N pivot:


BoR                  Found_pct                    K_mean         \
policy BoR DQN F1 DQN Fixed K=1   BoR DQN  F1 DQN Fixed K=1 BoR DQN F1 DQN   
N                                                                            
20       2.751  2.652     2.557    70.556  63.333    59.167   1.589  1.258   
50       3.655  3.480     3.339    74.722  63.889    59.167   2.631  1.222   
100      4.364  4.059     3.931    72.778  63.333    59.167   2.142  1.522   

                  
policy Fixed K=1  
N                 
20           1.0  
50           1.0  
100          1.0


Done.
Interpretation guide:
- If BoR is doing what you want, it should stay high on BoR as N grows.
- Difficulty buckets should show higher K on harder queries (4-10, 11+) than on easy ones (rank 1).
- If K barely changes across buckets, the policy is mostly learning a near-constant stop rule.


In [ ]:
# Self-contained Colab cell:
# MetaTool + sentence-transformer scorer + adaptive STOP/CONTINUE DQN
# Story-focused version:
#   1) varies candidate-set size N in {20, 50, 100}
#   2) uses HARD distractors only (no random fill)
#   3) reports difficulty-bucket behavior so adaptivity is visible
#
# Important experimental note:
#   The gold tool is ALWAYS included in the candidate set.
#   This isolates presentation-depth control rather than full-registry recall.
#
# Runtime:
#   CPU: a few minutes
#   GPU: faster
#
# Adjust MAX_QUERIES / TRAIN_EPISODES if needed.

import sys, subprocess, importlib.util, io, re, json, math, random, time
from collections import deque, defaultdict

def ensure(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

ensure("requests")
ensure("pandas")
ensure("numpy")
ensure("sklearn", "scikit-learn")
ensure("torch")
ensure("sentence_transformers", "sentence-transformers")

import requests
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from sentence_transformers import SentenceTransformer

try:
    from IPython.display import display
except:
    display = print

# -----------------------------
# Config
# -----------------------------
SEED = 42
MAX_QUERIES = 1200
MAX_PER_TOOL = 20
CANDIDATE_SIZES = [20, 50, 100]   # vary registry size
MODEL_NAME = "BAAI/bge-base-en-v1.5"

TRAIN_EPISODES = 5000
BATCH_SIZE = 128
REPLAY_SIZE = 50000
LR = 1e-3
GAMMA = 0.95
STEP_COST = 0.01
TARGET_UPDATE_EVERY = 500
PRINT_EVERY = 2500

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# Data URLs (MetaTool official repo)
# -----------------------------
CSV_URL = "https://raw.githubusercontent.com/HowieHwong/MetaTool/master/dataset/data/all_clean_data.csv"
TOOLS_URL = "https://raw.githubusercontent.com/HowieHwong/MetaTool/master/dataset/plugin_des.json"

# -----------------------------
# Helpers
# -----------------------------
def normalize_name(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).lower())

def pct(x):
    return f"{100*x:.1f}%"

def bor_reward(success, k, N):
    # exact for single-tool queries (R_q = 1): P_rand = k / N
    if not success:
        return 0.0
    p_rand = min(1.0, max(k / N, 1e-12))
    return max(0.0, -math.log2(p_rand))

def f1_reward(success, k):
    # single relevant tool:
    # F1 = 2/(k+1) if found else 0
    return 0.0 if not success else (2.0 / (k + 1.0))

def summarize_rows(name, rows):
    ks = np.array([r["k"] for r in rows], dtype=float)
    found = np.mean([r["success"] for r in rows]) if rows else 0.0
    bor = np.mean([r["bor"] for r in rows]) if rows else 0.0
    f1m = np.mean([r["f1"] for r in rows]) if rows else 0.0
    return {
        "policy": name,
        "K_mean": float(ks.mean()) if len(ks) else 0.0,
        "K_std": float(ks.std()) if len(ks) else 0.0,
        "Found": float(found),
        "BoR": float(bor),
        "F1": float(f1m),
        "n": len(rows),
    }

def difficulty_bucket(rank):
    if rank == 1:
        return "1"
    if 2 <= rank <= 3:
        return "2-3"
    if 4 <= rank <= 10:
        return "4-10"
    return "11+"

# -----------------------------
# Download + parse MetaTool
# -----------------------------
print("Downloading MetaTool...")
csv_text = requests.get(CSV_URL, timeout=120).text
tools_text = requests.get(TOOLS_URL, timeout=120).text

df = pd.read_csv(io.StringIO(csv_text), engine="python")
df.columns = [c.strip() for c in df.columns]

query_col = next((c for c in df.columns if c.lower() == "query"), None)
tool_col = next((c for c in df.columns if c.lower() == "tool"), None)
if query_col is None or tool_col is None:
    raise ValueError(f"Could not find Query/Tool columns. Found columns: {list(df.columns)}")

tool_desc_raw = json.loads(tools_text)

registry = {}
for tool_name, desc in tool_desc_raw.items():
    key = normalize_name(tool_name)
    if key and key not in registry:
        registry[key] = {
            "name": tool_name,
            "description": " ".join(str(desc).split())
        }

df = df[[query_col, tool_col]].copy()
df[query_col] = df[query_col].astype(str).str.strip()
df[tool_col] = df[tool_col].astype(str).str.strip()
df["tool_norm"] = df[tool_col].map(normalize_name)

df = df[df[query_col].str.len() > 0]
df = df[df["tool_norm"].isin(registry)]
df = df.drop_duplicates(subset=[query_col, "tool_norm"]).reset_index(drop=True)

# Balance across tools a bit
balanced_parts = []
for _, g in df.groupby("tool_norm"):
    n = min(len(g), MAX_PER_TOOL)
    balanced_parts.append(g.sample(n=n, random_state=SEED))
data = pd.concat(balanced_parts, ignore_index=True)

if len(data) > MAX_QUERIES:
    data = data.sample(n=MAX_QUERIES, random_state=SEED)

data = data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

tool_keys = sorted(registry.keys())
tool_names = [registry[k]["name"] for k in tool_keys]
tool_descs = [registry[k]["description"] for k in tool_keys]
tool_to_idx = {k: i for i, k in enumerate(tool_keys)}
FULL_REGISTRY_N = len(tool_keys)

print(f"MetaTool: {len(data)} queries, {FULL_REGISTRY_N} tools")
print("Candidate sets use HARD distractors only + guaranteed gold inclusion.")

# -----------------------------
# Sentence-transformer scorer
# -----------------------------
print(f"Loading sentence-transformer: {MODEL_NAME}")
st_model = SentenceTransformer(MODEL_NAME, device=device)

print("Encoding tool descriptions...")
tool_embs = st_model.encode(
    tool_descs,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Encoding queries...")
queries = data[query_col].tolist()
query_embs = st_model.encode(
    queries,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

# full score matrix: [num_queries, num_tools]
score_matrix = np.asarray(query_embs @ tool_embs.T, dtype=np.float32)

# -----------------------------
# Base instances over full registry
# -----------------------------
base_instances = []
for i, (_, row) in enumerate(data.iterrows()):
    gold_key = row["tool_norm"]
    gold_idx = tool_to_idx[gold_key]
    scores_all = score_matrix[i]
    ranked_global = np.argsort(-scores_all)
    gold_rank_full = int(np.where(ranked_global == gold_idx)[0][0] + 1)
    base_instances.append({
        "query": row[query_col],
        "gold_tool": registry[gold_key]["name"],
        "gold_idx": gold_idx,
        "scores_all": scores_all,
        "ranked_global": ranked_global,
        "gold_rank_full": gold_rank_full,
    })

# Example of full-registry scorer quality
print("\nFull-registry static baselines:")
for K in [1, 3, 5, 10, 20, 50, 100]:
    if K > FULL_REGISTRY_N:
        continue
    found = np.mean([inst["gold_rank_full"] <= K for inst in base_instances])
    p_rand = K / FULL_REGISTRY_N
    ceil = max(0.0, -math.log2(p_rand))
    print(f"  K={K:>3}: found={pct(found):>6}  P_rand={p_rand:>6.4f}  BoR_ceil={ceil:>4.1f} bits")

# -----------------------------
# Candidate-set construction
# -----------------------------
def make_candidate_instance(base_inst, N):
    """
    Hard-distractor candidate set:
      gold tool + top (N-1) highest-scoring negatives
    This makes the stopping problem genuinely harder than random-fill candidates.
    """
    gold_idx = base_inst["gold_idx"]
    ranked_global = base_inst["ranked_global"]
    hard_negatives = [tid for tid in ranked_global if tid != gold_idx][:N-1]
    cand = [gold_idx] + hard_negatives

    scores = np.asarray([base_inst["scores_all"][tid] for tid in cand], dtype=np.float32)
    order = np.argsort(-scores)
    ranked_tool_ids = [cand[j] for j in order]
    ranked_scores = scores[order]
    gold_rank = int(ranked_tool_ids.index(gold_idx) + 1)

    return {
        "query": base_inst["query"],
        "gold_tool": base_inst["gold_tool"],
        "gold_rank": gold_rank,
        "gold_rank_full": base_inst["gold_rank_full"],
        "ranked_tool_ids": ranked_tool_ids,
        "scores": ranked_scores,
        "N": N,
        "bucket": difficulty_bucket(gold_rank),
    }

# consistent split across all N
indices = np.arange(len(base_instances))
train_idx, test_idx = train_test_split(indices, test_size=0.30, random_state=SEED)
print(f"\nTrain: {len(train_idx)}, Test: {len(test_idx)}")
print(f"Step cost: {STEP_COST}, Gamma: {GAMMA} (same style as retrieval experiments)")

# -----------------------------
# RL state
# -----------------------------
def state_vec(inst, k):
    s = inst["scores"]
    N = inst["N"]
    idx = min(k - 1, N - 1)

    cur = float(s[idx])
    nxt = float(s[idx + 1]) if idx + 1 < N else float(s[idx])
    first = float(s[0])
    mean = float(s.mean())
    std = float(s.std() + 1e-6)
    gap = cur - nxt if idx + 1 < N else 0.0

    feats = np.array([
        k / N,
        math.log2(k + 1) / math.log2(N + 1),
        cur,
        nxt,
        gap,
        (cur - mean) / std,
        cur / (abs(first) + 1e-6),
    ], dtype=np.float32)
    return feats

# -----------------------------
# DQN
# -----------------------------
class QNet(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 2)   # 0=STOP, 1=CONTINUE
        )
    def forward(self, x):
        return self.net(x)

def train_dqn(train_instances, reward_name="bor"):
    reward_fn = bor_reward if reward_name == "bor" else f1_reward
    state_dim = len(state_vec(train_instances[0], 1))

    net = QNet(state_dim).to(device)
    target = QNet(state_dim).to(device)
    target.load_state_dict(net.state_dict())
    opt = optim.Adam(net.parameters(), lr=LR)
    replay = deque(maxlen=REPLAY_SIZE)

    epsilon_start, epsilon_end = 1.0, 0.05
    global_step = 0

    def greedy_action(state_np):
        with torch.no_grad():
            q = net(torch.tensor(state_np, dtype=torch.float32, device=device).unsqueeze(0))
            return int(q.argmax(dim=1).item())

    for ep in range(1, TRAIN_EPISODES + 1):
        inst = random.choice(train_instances)
        N = inst["N"]
        k = 1
        done = False
        epsilon = epsilon_end + (epsilon_start - epsilon_end) * max(0.0, 1.0 - ep / TRAIN_EPISODES)

        while not done:
            s = state_vec(inst, k)

            if random.random() < epsilon:
                a = random.randint(0, 1)
            else:
                a = greedy_action(s)

            if k >= N:
                a = 0

            if a == 0:  # STOP
                success = inst["gold_rank"] <= k
                terminal = reward_fn(success, k, inst["N"]) if reward_name == "bor" else reward_fn(success, k)
                r = float(terminal)
                replay.append((s, a, r, None, 1.0))
                done = True
            else:       # CONTINUE
                k2 = k + 1
                if k2 >= N:
                    # If continuing reaches the end, pay step cost then terminal reward at N
                    success = inst["gold_rank"] <= N
                    terminal = reward_fn(success, N, inst["N"]) if reward_name == "bor" else reward_fn(success, N)
                    r = float(-STEP_COST + terminal)
                    replay.append((s, a, r, None, 1.0))
                    done = True
                else:
                    s2 = state_vec(inst, k2)
                    r = float(-STEP_COST)
                    replay.append((s, a, r, s2, 0.0))
                    k = k2

            global_step += 1

            if len(replay) >= BATCH_SIZE:
                batch = random.sample(replay, BATCH_SIZE)

                states = torch.tensor(np.stack([b[0] for b in batch]), dtype=torch.float32, device=device)
                actions = torch.tensor([b[1] for b in batch], dtype=torch.long, device=device).unsqueeze(1)
                rewards = torch.tensor([b[2] for b in batch], dtype=torch.float32, device=device)
                dones = torch.tensor([b[4] for b in batch], dtype=torch.float32, device=device)

                nonfinal_mask = torch.tensor([b[3] is not None for b in batch], dtype=torch.bool, device=device)
                next_q = torch.zeros(BATCH_SIZE, dtype=torch.float32, device=device)

                if nonfinal_mask.any():
                    next_states = torch.tensor(
                        np.stack([b[3] for b in batch if b[3] is not None]),
                        dtype=torch.float32, device=device
                    )
                    with torch.no_grad():
                        next_q[nonfinal_mask] = target(next_states).max(dim=1).values

                q_values = net(states).gather(1, actions).squeeze(1)
                target_values = rewards + (1.0 - dones) * GAMMA * next_q

                loss = nn.SmoothL1Loss()(q_values, target_values)
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(), 5.0)
                opt.step()

            if global_step % TARGET_UPDATE_EVERY == 0:
                target.load_state_dict(net.state_dict())

        if ep % PRINT_EVERY == 0:
            print(f"Training {reward_name.upper()} DQN... {ep}/{TRAIN_EPISODES}")

    return net

# -----------------------------
# Evaluation
# -----------------------------
def rollout_model(inst, model):
    k = 1
    while True:
        s = state_vec(inst, k)
        with torch.no_grad():
            a = int(model(torch.tensor(s, dtype=torch.float32, device=device).unsqueeze(0)).argmax(dim=1).item())
        if a == 0 or k >= inst["N"]:
            break
        k += 1

    success = inst["gold_rank"] <= k
    return {
        "k": k,
        "success": success,
        "bor": bor_reward(success, k, inst["N"]),
        "f1": f1_reward(success, k),
        "bucket": inst["bucket"],
        "gold_rank": inst["gold_rank"],
    }

def eval_fixed(instances, K):
    rows = []
    for inst in instances:
        k = min(K, inst["N"])
        success = inst["gold_rank"] <= k
        rows.append({
            "k": k,
            "success": success,
            "bor": bor_reward(success, k, inst["N"]),
            "f1": f1_reward(success, k),
            "bucket": inst["bucket"],
            "gold_rank": inst["gold_rank"],
        })
    return rows

def eval_model(instances, model):
    return [rollout_model(inst, model) for inst in instances]

def choose_best_fixed_k(train_instances, ks):
    train_scores = []
    for K in ks:
        rows = eval_fixed(train_instances, K)
        train_scores.append((K, np.mean([r["bor"] for r in rows])))
    best_k, best_bor = max(train_scores, key=lambda x: x[1])
    return best_k, train_scores

def bucket_table(rows_dict):
    # rows_dict: policy_name -> rows
    all_rows = []
    bucket_order = ["1", "2-3", "4-10", "11+"]
    for bucket in bucket_order:
        row_out = {"bucket": bucket}
        present = False
        for name, rows in rows_dict.items():
            sub = [r for r in rows if r["bucket"] == bucket]
            if len(sub) > 0:
                present = True
                stats = summarize_rows(name, sub)
                row_out[f"{name}_n"] = stats["n"]
                row_out[f"{name}_K"] = round(stats["K_mean"], 2)
                row_out[f"{name}_Found"] = round(100 * stats["Found"], 1)
                row_out[f"{name}_BoR"] = round(stats["BoR"], 2)
            else:
                row_out[f"{name}_n"] = 0
                row_out[f"{name}_K"] = None
                row_out[f"{name}_Found"] = None
                row_out[f"{name}_BoR"] = None
        if present:
            all_rows.append(row_out)
    return pd.DataFrame(all_rows)

# -----------------------------
# Main experiment loop
# -----------------------------
overall_rows = []
bucket_tables = {}

for N in CANDIDATE_SIZES:
    print("\n" + "=" * 90)
    print(f"CANDIDATE SET SIZE N = {N} (hard distractors only)")
    print("=" * 90)

    instances_N = [make_candidate_instance(inst, N) for inst in base_instances]
    train_insts = [instances_N[i] for i in train_idx]
    test_insts  = [instances_N[i] for i in test_idx]

    # static baselines for this N
    print("Static baselines:")
    static_ks = [1, 3, 5, 10, 20, 50, N]
    static_ks = [k for k in static_ks if k <= N]
    for K in static_ks:
        found = np.mean([inst["gold_rank"] <= K for inst in test_insts])
        p_rand = K / N
        ceil = max(0.0, -math.log2(p_rand))
        print(f"  K={K:>3}: found={pct(found):>6}  P_rand={p_rand:>6.4f}  BoR_ceil={ceil:>4.1f} bits")

    # difficulty distribution
    bucket_counts = pd.Series([inst["bucket"] for inst in test_insts]).value_counts().reindex(["1","2-3","4-10","11+"], fill_value=0)
    print("Test difficulty buckets:", dict(bucket_counts))

    # choose best fixed-K baseline on TRAIN by BoR
    best_fixed_k, train_fixed_scores = choose_best_fixed_k(train_insts, static_ks)
    print(f"Best fixed K on TRAIN by BoR: K={best_fixed_k}")

    # train models
    t0 = time.time()
    bor_model = train_dqn(train_insts, reward_name="bor")
    f1_model  = train_dqn(train_insts, reward_name="f1")
    print(f"Training time for N={N}: {(time.time()-t0):.1f}s on {device}")

    # eval
    bor_rows = eval_model(test_insts, bor_model)
    f1_rows = eval_model(test_insts, f1_model)
    best_fixed_rows = eval_fixed(test_insts, best_fixed_k)
    fixed1_rows = eval_fixed(test_insts, 1)

    # overall summary
    summary_df = pd.DataFrame([
        summarize_rows("BoR DQN", bor_rows),
        summarize_rows("F1 DQN", f1_rows),
        summarize_rows(f"Fixed K={best_fixed_k}", best_fixed_rows),
        summarize_rows("Fixed K=1", fixed1_rows),
    ])
    summary_df["N"] = N
    overall_rows.extend(summary_df.to_dict("records"))

    print(f"\nOVERALL RESULTS @ N={N}")
    display(summary_df[["N","policy","n","K_mean","K_std","Found","BoR","F1"]].round(3))

    # difficulty buckets
    bt = bucket_table({
        "BoR": bor_rows,
        "F1": f1_rows,
        f"K{best_fixed_k}": best_fixed_rows,
    })
    bucket_tables[N] = bt

    print(f"\nDIFFICULTY BUCKETS @ N={N}")
    display(bt)

    # sample decisions
    print(f"\nSample BoR decisions @ N={N}:")
    sample = bor_rows[:6]
    for inst, row in zip(test_insts[:6], sample):
        mark = "✓" if row["success"] else "✗"
        print(f"  K={row['k']:>3} {mark} rank={inst['gold_rank']:>2} bucket={inst['bucket']:<4} | {inst['query'][:90]}")

# -----------------------------
# Cross-N summary
# -----------------------------
cross_df = pd.DataFrame(overall_rows)
cross_df["Found_pct"] = 100 * cross_df["Found"]

print("\n" + "#" * 90)
print("CROSS-N SUMMARY")
print("#" * 90)
display(
    cross_df[["N","policy","K_mean","K_std","Found_pct","BoR","F1"]]
    .sort_values(["N","policy"])
    .round(3)
)

# Small pivot for readability
pivot = cross_df.pivot_table(index="N", columns="policy", values=["K_mean","Found_pct","BoR"], aggfunc="first")
print("\nCompact cross-N pivot:")
display(pivot.round(3))

print("\nDone.")
print("Interpretation guide:")
print("- If BoR is doing what you want, it should stay high on BoR as N grows.")
print("- Difficulty buckets should show higher K on harder queries (4-10, 11+) than on easy ones (rank 1).")
print("- If K barely changes across buckets, the policy is mostly learning a near-constant stop rule.")

MetaTool: 1200 queries, 199 tools
Candidate sets use HARD distractors only + guaranteed gold inclusion.
Loading sentence-transformer: BAAI/bge-base-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...


Sample BoR decisions @ N=100:
  K=  1 ✓ rank= 1 bucket=1    | I want to make some changes to my FPL team's defense. Which defenders should I consider?
  K=  1 ✓ rank= 1 bucket=1    | What font does this company use?
  K=  1 ✓ rank= 1 bucket=1    | It's so frustrating to search for international job listings. I need your assistance in fi
  K=  1 ✓ rank= 1 bucket=1    | Can you search for any art pieces related to Asian culture from The Metropolitan Museum of
  K=  1 ✓ rank= 1 bucket=1    | Hey Chatbot, I have a blog and I want to monetize my outgoing traffic. Can you help me wit
  K=  1 ✓ rank= 1 bucket=1    | I just started playing this life simulator and I am feeling lost. Can you guide me?

##########################################################################################
CROSS-N SUMMARY
##########################################################################################


,N,policy,K_mean,K_std,Found_pct,BoR,F1
0,20,BoR DQN,1.719,0.929,66.667,2.587,0.572
1,20,F1 DQN,1.189,0.451,60.833,2.559,0.585
2,20,Fixed K=1,1.000,0.000,57.222,2.473,0.572
3,20,Fixed K=1,1.000,0.000,57.222,2.473,0.572
4,50,BoR DQN,2.436,2.089,70.556,3.509,0.564
5,50,F1 DQN,1.225,0.524,61.111,3.350,0.579
6,50,Fixed K=1,1.000,0.000,57.222,3.230,0.572
7,50,Fixed K=1,1.000,0.000,57.222,3.230,0.572
8,100,BoR DQN,2.397,1.762,71.389,4.241,0.563
9,100,F1 DQN,1.222,0.435,61.389,3.961,0.575



Compact cross-N pivot:


BoR                  Found_pct                    K_mean         \
policy BoR DQN F1 DQN Fixed K=1   BoR DQN  F1 DQN Fixed K=1 BoR DQN F1 DQN   
N                                                                            
20       2.587  2.559     2.473    66.667  60.833    57.222   1.719  1.189   
50       3.509  3.350     3.230    70.556  61.111    57.222   2.436  1.225   
100      4.241  3.961     3.802    71.389  61.389    57.222   2.397  1.222   

                  
policy Fixed K=1  
N                 
20           1.0  
50           1.0  
100          1.0


Done.
Interpretation guide:
- If BoR is doing what you want, it should stay high on BoR as N grows.
- Difficulty buckets should show higher K on harder queries (4-10, 11+) than on easy ones (rank 1).
- If K barely changes across buckets, the policy is mostly learning a near-constant stop rule.
